# DICE ITC Results Notebook

This notebook is the public, portable entry point for reproducing the DICE ITC results. Run it top to bottom.

It regenerates:
- the core analysis tables and figures
- the full DICE paper results
- the workload-holdout appendix results
- the `results_itc_paper/` and `results_itc_appendix/` bundles
- the reproducibility manifest

The notebook is organized in paper order:
1. execution guard and end-to-end run
2. benign/anomaly separation and core full-pipeline metrics
3. calibrated reliability without per-workload tuning
4. digital-twin observability, tier, and mechanism dashboards
5. workload-holdout robustness, bootstrap confidence intervals, and reproducibility artifacts

Methodology reflected here:
1. Benign-only regime-conditioned micro-twin heads across Tier-0, Tier-0/1, and Tier-0/1/2 observation availability.
2. Online residualization with fixed block summaries.
3. Sequential conformal decisioning with persistent alerts.
4. Mechanism-level diagnosis from grouped residual evidence.
5. Workload-holdout robustness as a portable workload/software-drift proxy.
6. Reduced-observability robustness across tier subsets.


In [ ]:
from pathlib import Path
import json
import os
import sys
import tempfile
from time import perf_counter

import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## Notebook-Local Helper Functions

This notebook is self-contained. The next cell defines the helper functions used later for tier dashboards, bootstrap confidence intervals, paper-ready composite figures, and optional LLM-ready case cards.


In [ ]:
CFG_LABEL = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}


def _cfg_labels(values: pd.Series) -> list[str]:
    return [CFG_LABEL.get(str(v), str(v)) for v in values]


def render_tier_correlation_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    tier_case = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    tier_final = tier_case[tier_case["config"] == "tier0_tier1_tier2"].copy()
    tier_cols = ["tier0_share", "tier1_alt_share", "tier2_share"]

    tier_corr = tier_final[tier_cols].corr().round(4)
    tier_corr.to_csv(paper_full / "tier_share_correlation.csv")

    stressor_tier = tier_final.groupby("stressor", sort=False)[tier_cols].mean().reset_index()
    stressor_tier.to_csv(paper_full / "stressor_tier_share_summary.csv", index=False)

    tier_final["ternary_x"] = tier_final["tier1_alt_share"] + 0.5 * tier_final["tier2_share"]
    tier_final["ternary_y"] = (np.sqrt(3.0) / 2.0) * tier_final["tier2_share"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    im = axes[0].imshow(tier_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_xticks(range(3), ["Tier-0", "Tier-1", "Tier-2"], rotation=30, ha="right")
    axes[0].set_yticks(range(3), ["Tier-0", "Tier-1", "Tier-2"])
    axes[0].set_title("Tier-share correlation")
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f"{tier_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(stressor_tier))
    axes[1].bar(x, stressor_tier["tier0_share"], label="Tier-0")
    axes[1].bar(x, stressor_tier["tier1_alt_share"], bottom=stressor_tier["tier0_share"], label="Tier-1")
    axes[1].bar(
        x,
        stressor_tier["tier2_share"],
        bottom=stressor_tier["tier0_share"] + stressor_tier["tier1_alt_share"],
        label="Tier-2",
    )
    axes[1].set_xticks(x, stressor_tier["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Mean tier evidence by stressor")
    axes[1].legend(loc="upper right")

    triangle = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.5, np.sqrt(3.0) / 2.0],
            [0.0, 0.0],
        ]
    )
    axes[2].plot(triangle[:, 0], triangle[:, 1], color="black")
    for stressor, d in tier_final.groupby("stressor", sort=False):
        axes[2].scatter(d["ternary_x"], d["ternary_y"], s=36, alpha=0.8, label=stressor)
    axes[2].text(-0.04, -0.03, "Tier-0")
    axes[2].text(1.01, -0.03, "Tier-1")
    axes[2].text(0.46, np.sqrt(3.0) / 2.0 + 0.03, "Tier-2")
    axes[2].set_title("Per-case tier composition")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
    axes[2].legend(loc="upper right", fontsize=7)

    fig.tight_layout()
    png = paper_fig / "fig_tier_correlation_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_corr, stressor_tier, png


def render_bootstrap_confidence(
    case_pred: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
    samples: int = 1000,
    seed: int = 0,
) -> tuple[pd.DataFrame, Path]:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for cfg, d in case_pred.groupby("config", sort=False):
        stats: list[dict[str, float]] = []
        for _ in range(samples):
            sample = d.sample(n=len(d), replace=True, random_state=int(rng.integers(1 << 32)))
            if sample["label"].nunique() < 2:
                continue
            benign = sample[sample["label"] == 0]
            anomaly = sample[sample["label"] == 1]
            stats.append(
                {
                    "roc_auc_wc": roc_auc_score(sample["label"], sample["run_score_wc"]),
                    "pr_auc_wc": average_precision_score(sample["label"], sample["run_score_wc"]),
                    "benign_run_false_alarm_rate": benign["run_alert"].mean(),
                    "anomaly_run_detection_rate": anomaly["run_alert"].mean(),
                    "median_time_to_detect_s": anomaly.loc[
                        anomaly["run_alert"] == 1, "time_to_detect_s"
                    ].median(),
                }
            )

        boot = pd.DataFrame(stats)
        rows.append(
            {
                "config": cfg,
                "roc_auc_wc_lo": boot["roc_auc_wc"].quantile(0.025),
                "roc_auc_wc_hi": boot["roc_auc_wc"].quantile(0.975),
                "pr_auc_wc_lo": boot["pr_auc_wc"].quantile(0.025),
                "pr_auc_wc_hi": boot["pr_auc_wc"].quantile(0.975),
                "fpr_lo": boot["benign_run_false_alarm_rate"].quantile(0.025),
                "fpr_hi": boot["benign_run_false_alarm_rate"].quantile(0.975),
                "detect_lo": boot["anomaly_run_detection_rate"].quantile(0.025),
                "detect_hi": boot["anomaly_run_detection_rate"].quantile(0.975),
                "ttd_lo": boot["median_time_to_detect_s"].quantile(0.025),
                "ttd_hi": boot["median_time_to_detect_s"].quantile(0.975),
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(paper_full / "bootstrap_confidence_intervals.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(out))

    pr_mid = (out["pr_auc_wc_lo"] + out["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - out["pr_auc_wc_lo"], out["pr_auc_wc_hi"] - pr_mid])
    axes[0].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4)
    axes[0].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[0].set_title("Bootstrap AUC-PR CI")

    fpr_mid = (out["fpr_lo"] + out["fpr_hi"]) / 2.0
    fpr_err = np.vstack([fpr_mid - out["fpr_lo"], out["fpr_hi"] - fpr_mid])
    axes[1].errorbar(x, fpr_mid, yerr=fpr_err, fmt="o", capsize=4)
    axes[1].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[1].set_title("Bootstrap benign-FPR CI")

    ttd_mid = (out["ttd_lo"] + out["ttd_hi"]) / 2.0
    ttd_err = np.vstack([ttd_mid - out["ttd_lo"], out["ttd_hi"] - ttd_mid])
    axes[2].errorbar(x, ttd_mid, yerr=ttd_err, fmt="o", capsize=4)
    axes[2].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[2].set_title("Bootstrap time-to-detect CI")

    fig.tight_layout()
    png = paper_fig / "fig_bootstrap_confidence_intervals.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out, png


def export_llm_case_cards(
    out_full: Path,
    appendix_full: Path,
) -> pd.DataFrame:
    case_diag = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    cards = case_diag[case_diag["config"] == "tier0_tier1_tier2"].copy()
    cards = cards[
        [
            "case_id",
            "workload",
            "stressor",
            "dominant_tier",
            "dominant_mechanism",
            "top_feature_1",
            "top_feature_score_1",
            "top_mechanism_1",
            "top_mechanism_score_1",
        ]
    ].copy()
    cards["llm_summary_prompt"] = (
        "Summarize this DICE anomaly case for a reviewer. "
        "Case=" + cards["case_id"]
        + "; workload=" + cards["workload"]
        + "; stressor=" + cards["stressor"]
        + "; dominant tier=" + cards["dominant_tier"]
        + "; dominant mechanism=" + cards["dominant_mechanism"]
        + "; top feature=" + cards["top_feature_1"].astype(str)
        + "; top mechanism=" + cards["top_mechanism_1"].astype(str)
    )
    cards.to_csv(appendix_full / "llm_case_cards.csv", index=False)
    return cards


def render_paper_performance_stack(
    case_pred: pd.DataFrame,
    overall_full: pd.DataFrame,
    sequential: pd.DataFrame,
    reliability: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    summary = (
        overall_full[
            [
                "config",
                "roc_auc",
                "pr_auc",
                "roc_auc_wc",
                "pr_auc_wc",
                "median_nominal_score_wc",
                "median_anomaly_score_wc",
            ]
        ]
        .merge(
            sequential[
                [
                    "config",
                    "anomaly_detect_rate",
                    "median_time_to_detect_s",
                ]
            ],
            on="config",
        )
        .merge(
            reliability[
                [
                    "config",
                    "target_alpha",
                    "benign_block_false_alarm_rate",
                ]
            ],
            on="config",
        )
    )
    summary["config_label"] = _cfg_labels(summary["config"])
    summary["reliability_margin"] = summary["target_alpha"] - summary["benign_block_false_alarm_rate"]
    summary.to_csv(paper_full / "paper_performance_stack_summary.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    final = case_pred[case_pred["config"] == "tier0_tier1_tier2"].copy()
    benign = final[final["label"] == 0]["run_score_wc"].to_numpy(dtype=float)
    anomaly = final[final["label"] == 1]["run_score_wc"].to_numpy(dtype=float)
    bp = axes[0, 0].boxplot([benign, anomaly], labels=["Benign", "Anomaly"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2563eb", "#dc2626"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[0, 0].set_title("A. Final-head score separation")
    axes[0, 0].set_ylabel("Workload-conditioned run score")

    base_color = "#94a3b8"
    wc_color = "#0f766e"
    for _, row in summary.iterrows():
        axes[0, 1].scatter(row["roc_auc"], row["pr_auc"], color=base_color, s=70)
        axes[0, 1].scatter(row["roc_auc_wc"], row["pr_auc_wc"], color=wc_color, s=90)
        axes[0, 1].annotate(
            row["config_label"],
            (row["roc_auc_wc"], row["pr_auc_wc"]),
            textcoords="offset points",
            xytext=(6, 6),
        )
        axes[0, 1].plot([row["roc_auc"], row["roc_auc_wc"]], [row["pr_auc"], row["pr_auc_wc"]], color="#475569")
    axes[0, 1].set_xlabel("Run-level ROC-AUC")
    axes[0, 1].set_ylabel("Run-level Average Precision")
    axes[0, 1].set_title("B. Digital-twin score refinement")

    x = np.arange(len(summary))
    axes[1, 0].bar(x, summary["benign_block_false_alarm_rate"], color="#f59e0b")
    axes[1, 0].axhline(float(summary["target_alpha"].iloc[0]), color="black", linestyle="--", linewidth=1.2)
    axes[1, 0].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 0].set_ylabel("Empirical benign block FAR")
    axes[1, 0].set_title("C. Calibrated reliability")

    bars = axes[1, 1].bar(x, summary["anomaly_detect_rate"], color="#16a34a", label="Detection rate")
    ax2 = axes[1, 1].twinx()
    ax2.plot(x, summary["median_time_to_detect_s"], color="#1d4ed8", marker="o", linewidth=2, label="Median TTD")
    axes[1, 1].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 1].set_ylabel("Run-level detection rate")
    ax2.set_ylabel("Median time-to-detect (s)")
    axes[1, 1].set_title("D. Operational decision performance")
    axes[1, 1].legend([bars], ["Detection rate"], loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_performance_stack.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return summary, png


def render_explainability_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Path]:
    tier_contrib = pd.read_csv(out_full / "stressor_tier_contributions.csv")
    mechanism = pd.read_csv(out_full / "mechanism_group_summary.csv")
    cm = pd.read_csv(out_full / "stressor_confusion_matrix.csv", index_col=0)

    tier_contrib.to_csv(paper_full / "paper_tier_contribution_summary.csv", index=False)
    mechanism.to_csv(paper_full / "paper_mechanism_summary.csv", index=False)
    cm.to_csv(paper_full / "paper_stressor_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    x = np.arange(len(tier_contrib))
    axes[0].bar(x, tier_contrib["tier0_share"], label="Tier-0")
    axes[0].bar(x, tier_contrib["tier1_alt_share"], bottom=tier_contrib["tier0_share"], label="Tier-1")
    axes[0].bar(
        x,
        tier_contrib["tier2_share"],
        bottom=tier_contrib["tier0_share"] + tier_contrib["tier1_alt_share"],
        label="Tier-2",
    )
    axes[0].set_xticks(x, tier_contrib["stressor"], rotation=30, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("A. Tier contribution by stressor")
    axes[0].legend(loc="upper right")

    mech_cols = [
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
    ]
    bottom = np.zeros(len(mechanism))
    colors = ["#0f766e", "#1d4ed8", "#dc2626", "#9333ea", "#b45309"]
    for col, color in zip(mech_cols, colors):
        axes[1].bar(np.arange(len(mechanism)), mechanism[col], bottom=bottom, label=col.replace("_share", ""), color=color)
        bottom += mechanism[col].to_numpy(dtype=float)
    axes[1].set_xticks(np.arange(len(mechanism)), mechanism["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("B. Mechanism evidence by stressor")
    axes[1].legend(loc="upper right", fontsize=7)

    im = axes[2].imshow(cm.values, cmap="Blues")
    axes[2].set_xticks(range(len(cm.columns)), list(cm.columns), rotation=30, ha="right")
    axes[2].set_yticks(range(len(cm.index)), list(cm.index))
    axes[2].set_title("C. Stressor diagnosis confusion")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    png = paper_fig / "fig_paper_explainability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_contrib, mechanism, cm, png


def render_portability_dashboard(
    frontier: pd.DataFrame,
    holdout: pd.DataFrame,
    bootstrap_ci: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    view = frontier.copy()
    view["config_label"] = _cfg_labels(view["config"])
    holdout_view = holdout.copy()
    holdout_view["config_label"] = _cfg_labels(holdout_view["config"])
    boot_view = bootstrap_ci.copy()
    boot_view["config_label"] = _cfg_labels(boot_view["config"])

    portability_summary = view[
        [
            "config",
            "config_label",
            "n_features",
            "portable_pr_auc",
            "holdout_worst_pr_auc",
            "reliability_margin",
            "joint_detection_diagnosis",
        ]
    ].copy()
    portability_summary.to_csv(paper_full / "paper_portability_summary.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    scatter = axes[0].scatter(
        view["n_features"],
        view["portable_pr_auc"],
        s=view["joint_detection_diagnosis"].fillna(0.0) * 1800 + 140,
        c=view["reliability_margin"],
        cmap="viridis",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in view.iterrows():
        axes[0].annotate(row["config_label"], (row["n_features"], row["portable_pr_auc"]), textcoords="offset points", xytext=(6, 6))
    axes[0].set_xlabel("Median active features")
    axes[0].set_ylabel("Portable AUC-PR")
    axes[0].set_title("A. Observability-portability frontier")
    fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(holdout_view))
    width = 0.35
    axes[1].bar(x - width / 2.0, holdout_view["mean_pr_auc"], width=width, label="Mean holdout PR")
    axes[1].bar(x + width / 2.0, holdout_view["worst_pr_auc"], width=width, label="Worst holdout PR")
    axes[1].set_xticks(x, holdout_view["config_label"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title("B. Holdout portability")
    axes[1].legend(loc="upper right")

    x = np.arange(len(boot_view))
    pr_mid = (boot_view["pr_auc_wc_lo"] + boot_view["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - boot_view["pr_auc_wc_lo"], boot_view["pr_auc_wc_hi"] - pr_mid])
    axes[2].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4, color="#1d4ed8", label="AP CI")
    det_mid = (boot_view["detect_lo"] + boot_view["detect_hi"]) / 2.0
    det_err = np.vstack([det_mid - boot_view["detect_lo"], boot_view["detect_hi"] - det_mid])
    axes[2].errorbar(x, det_mid, yerr=det_err, fmt="o", capsize=4, color="#16a34a", label="Detect-rate CI")
    axes[2].set_xticks(x, boot_view["config_label"], rotation=30, ha="right")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("C. Bootstrap uncertainty")
    axes[2].legend(loc="lower right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_portability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return portability_summary, png


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / 'data generation').exists() and (base / 'environment.yml').exists():
            return base
    raise RuntimeError('Could not locate the DICE repository root from the current working directory.')


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Execution Guard

The notebook should be launched from the intended DICE clone, not from a stale copy in `Trash` or another transient folder.

The next cell validates the repository location, confirms the released dataset is available, and prepares the main paper and appendix output folders.


In [ ]:
if '.Trash' in str(REPO_ROOT):
    raise RuntimeError(
        'This notebook was launched from a Trash clone. Reopen it from your intended DICE repository checkout.'
    )
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

PAPER_FULL = OUT_PAPER / 'full'
PAPER_FIG = PAPER_FULL / 'figures'
APPENDIX_FULL = OUT_APPENDIX / 'full'
NOTEBOOK_RUNTIME = OUT_PAPER / 'runtime_summary.json'

for path in [OUT_PAPER, OUT_APPENDIX, PAPER_FULL, PAPER_FIG, APPENDIX_FULL]:
    path.mkdir(parents=True, exist_ok=True)

print('Validated repository root :', REPO_ROOT)
print('Validated dataset root    :', DATASET_ROOT)
print('Main paper output folder  :', PAPER_FULL)
print('Appendix output folder    :', APPENDIX_FULL)


## Notebook-Local Pipeline Engine

This section **defines** the notebook-local backend. It does not run any experiments yet.

What this section provides:
- the tier-level analysis pipeline
- the full benign-trained DICE digital-twin pipeline
- the notebook-local orchestration function `run_notebook_pipeline(...)`

If you are looking for the cell that actually runs the digital twin, go to **Run End-to-End** below.


In [ ]:
import hashlib
import importlib.metadata
import platform
import types
from datetime import datetime, timezone

NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE = '#!/usr/bin/env python3\n"""\nGenerate paper-ready Results/Analysis artifacts from DICE tiered dataset.\n\nOutputs:\n- CSV tables (overall metrics, stressor metrics, workload summaries, feature inventory)\n- LaTeX tables ready for Overleaf\n- PNG figures (AF-index trajectories, separability heatmaps, score distributions)\n- Markdown summary with key values to paste into paper draft\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Tuple\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nIGNORE_COLS = {\n    "idx",\n    "ts_unix_s",\n    "t_rel_s",\n    "timestamp",\n    "time",\n    "ts",\n}\n\nTIER_PRETTY = {\n    "tier0": "Tier-0",\n    "tier1_alt": "Tier-1",\n    "tier2": "Tier-2",\n}\n\nCOLOR_BY_STRESSOR = {\n    "NOMINAL": "#000000",\n    "ATOMIC": "#e57373",\n    "BRANCH": "#66bb6a",\n    "CACHE": "#f6a04d",\n    "MEMBW": "#b39ddb",\n    "TLB": "#bcaaa4",\n}\n\n\n@dataclass(frozen=True)\nclass TierData:\n    tier: str\n    features: List[str]\n    run_df: pd.DataFrame\n    timeseries: Dict[str, Dict[str, np.ndarray]]\n    case_quality: pd.DataFrame\n    union_features: List[str]\n\n\ndef case_id(workload: str, stressor: str) -> str:\n    return f"{workload}__{stressor}"\n\n\ndef robust_scale(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, y_score))\n\n\ndef safe_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, y_score))\n\n\ndef downsample_to_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    out = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return out.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef read_case_csv(root: Path, tier: str, workload: str, stressor: str) -> pd.DataFrame:\n    p = root / tier / case_id(workload, stressor) / TIER_FILE[tier]\n    if not p.exists():\n        raise FileNotFoundError(f"Missing case file: {p}")\n    df = pd.read_csv(p)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_feature_columns(df: pd.DataFrame) -> List[str]:\n    cols = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            cols.append(c)\n    return cols\n\n\ndef discover_features(root: Path, tier: str, source_hz: int = 5) -> Tuple[List[str], List[str], pd.DataFrame]:\n    common = None\n    union = set()\n    rows = []\n    for w in WORKLOADS:\n        for s in STRESSORS:\n            p = root / tier / case_id(w, s) / TIER_FILE[tier]\n            df = pd.read_csv(p)\n            cols = set(numeric_feature_columns(df))\n            union |= cols\n            common = cols if common is None else (common & cols)\n            rows.append(\n                {\n                    "tier": tier,\n                    "case_id": case_id(w, s),\n                    "workload": w,\n                    "stressor": s,\n                    "rows_5hz": int(len(df)),\n                    "cols_total": int(df.shape[1]),\n                    "numeric_cols": int(len(cols)),\n                    "nan_fraction": float(df.isna().mean().mean()),\n                    "file_bytes": int(p.stat().st_size),\n                }\n            )\n    common_list = sorted(common) if common else []\n    union_list = sorted(union)\n\n    # Drop globally near-constant channels from common list.\n    keep = []\n    for f in common_list:\n        vals = []\n        for w in WORKLOADS:\n            for s in STRESSORS:\n                d = downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz)\n                vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep, union_list, pd.DataFrame(rows)\n\n\ndef build_tier_data(root: Path, tier: str, source_hz: int = 5) -> TierData:\n    features, union_features, quality = discover_features(root, tier, source_hz=source_hz)\n    run_rows = []\n    timeseries = {w: {} for w in WORKLOADS}\n\n    for w in WORKLOADS:\n        ds = {s: downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz) for s in STRESSORS}\n        n = min(len(v) for v in ds.values())\n        arr = {\n            s: ds[s].iloc[:n][features].to_numpy(dtype=float, copy=True)\n            for s in STRESSORS\n        }\n        baseline = arr["NOMINAL"]\n        med = np.nanmedian(baseline, axis=0)\n        scale = np.array([robust_scale(baseline[:, j]) for j in range(baseline.shape[1])], dtype=float)\n        scale[scale <= 1e-12] = 1.0\n\n        for s in STRESSORS:\n            z = np.abs((arr[s] - med) / (scale + 1e-12))\n            score_ts = np.nanmean(z, axis=1)\n            timeseries[w][s] = score_ts\n            run_rows.append(\n                {\n                    "tier": tier,\n                    "workload": w,\n                    "stressor": s,\n                    "label": 0 if s == "NOMINAL" else 1,\n                    "run_score_median": float(np.nanmedian(score_ts)),\n                    "run_score_mean": float(np.nanmean(score_ts)),\n                    "run_score_p95": float(np.nanpercentile(score_ts, 95)),\n                    "samples_1hz": int(len(score_ts)),\n                }\n            )\n\n    return TierData(\n        tier=tier,\n        features=features,\n        run_df=pd.DataFrame(run_rows),\n        timeseries=timeseries,\n        case_quality=quality,\n        union_features=union_features,\n    )\n\n\ndef make_overall_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        y = df["label"].to_numpy(dtype=int)\n        s = df["run_score_median"].to_numpy(dtype=float)\n\n        nom = df[df["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = df[df["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        tau95 = float(np.quantile(nom, 0.95))\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": int(len(td.features)),\n                "n_features_union": int(len(td.union_features)),\n                "n_cases": int(len(df)),\n                "roc_auc": safe_auc(y, s),\n                "pr_auc": safe_ap(y, s),\n                "median_nominal": float(np.median(nom)),\n                "median_anomaly": float(np.median(anm)),\n                "anom_nom_ratio": float(np.median(anm) / (np.median(nom) + 1e-12)),\n                "threshold_q95_nominal": tau95,\n                "fpr_at_q95": float(np.mean(nom > tau95)),\n                "tpr_at_q95": float(np.mean(anm > tau95)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef make_stressor_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        neg = df[df["stressor"] == "NOMINAL"][["workload", "run_score_median"]].set_index("workload")\n        for a in ANOMALIES:\n            pos = df[df["stressor"] == a][["workload", "run_score_median"]].set_index("workload")\n            merged = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n            y_true = np.array([0] * len(merged) + [1] * len(merged), dtype=int)\n            y_score = np.concatenate(\n                [\n                    merged["run_score_median_neg"].to_numpy(dtype=float),\n                    merged["run_score_median_pos"].to_numpy(dtype=float),\n                ]\n            )\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "stressor": a,\n                    "n_pos": int(len(merged)),\n                    "n_neg": int(len(merged)),\n                    "roc_auc": safe_auc(y_true, y_score),\n                    "pr_auc": safe_ap(y_true, y_score),\n                    "median_neg": float(np.median(merged["run_score_median_neg"])),\n                    "median_pos": float(np.median(merged["run_score_median_pos"])),\n                    "pos_neg_ratio": float(\n                        np.median(merged["run_score_median_pos"])\n                        / (np.median(merged["run_score_median_neg"]) + 1e-12)\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_workload_summary(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        for w in WORKLOADS:\n            d = df[df["workload"] == w]\n            nom = d[d["stressor"] == "NOMINAL"]["run_score_median"].iloc[0]\n            anm = d[d["stressor"] != "NOMINAL"]["run_score_median"].to_numpy(dtype=float)\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "workload": w,\n                    "nominal_score": float(nom),\n                    "anomaly_median_score": float(np.median(anm)),\n                    "anomaly_nominal_ratio": float(np.median(anm) / (float(nom) + 1e-12)),\n                    "anomaly_p95_score": float(np.percentile(anm, 95)),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef table_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:\n    rendered = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{rendered}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef save_metric_tables(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    workload: pd.DataFrame,\n    features: pd.DataFrame,\n    quality: pd.DataFrame,\n    runs: pd.DataFrame,\n) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    overall_out = overall.sort_values("tier")\n    stressor_out = stressor.sort_values(["tier", "stressor"])\n    workload_out = workload.sort_values(["tier", "workload"])\n    features_out = features.sort_values("tier")\n    quality_out = quality.sort_values(["tier", "case_id"])\n    runs_out = runs.sort_values(["tier", "workload", "stressor"])\n\n    overall_out.to_csv(out_dir / "table_overall_metrics.csv", index=False)\n    stressor_out.to_csv(out_dir / "table_stressor_metrics.csv", index=False)\n    workload_out.to_csv(out_dir / "table_workload_summary.csv", index=False)\n    features_out.to_csv(out_dir / "table_feature_inventory.csv", index=False)\n    quality_out.to_csv(out_dir / "table_case_quality.csv", index=False)\n    runs_out.to_csv(out_dir / "table_run_scores.csv", index=False)\n\n    overall_tex = overall_out[\n        [\n            "tier_name",\n            "n_features_common",\n            "roc_auc",\n            "pr_auc",\n            "median_nominal",\n            "median_anomaly",\n            "anom_nom_ratio",\n        ]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "n_features_common": "Common Features",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "median_nominal": "Median(Nominal)",\n            "median_anomaly": "Median(Anomaly)",\n            "anom_nom_ratio": "Anomaly/Nominal",\n        }\n    )\n\n    stressor_tex = stressor_out[\n        ["tier_name", "stressor", "roc_auc", "pr_auc", "pos_neg_ratio"]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n        }\n    )\n\n    (out_dir / "table_overall_metrics.tex").write_text(\n        table_to_latex(\n            overall_tex,\n            "Run-level anomaly separability by telemetry tier (AF-index score).",\n            "tab:dice_overall_metrics",\n        )\n    )\n    (out_dir / "table_stressor_metrics.tex").write_text(\n        table_to_latex(\n            stressor_tex,\n            "Per-stressor separability by tier (four workloads pooled per stressor).",\n            "tab:dice_stressor_metrics",\n        )\n    )\n\n\ndef plot_af_timeseries(out_dir: Path, tier_data: Iterable[TierData]) -> List[str]:\n    out_paths = []\n    for td in tier_data:\n        fig, axes = plt.subplots(len(WORKLOADS), 1, figsize=(16, 13), sharex=True)\n        if len(WORKLOADS) == 1:\n            axes = [axes]\n\n        for i, w in enumerate(WORKLOADS):\n            ax = axes[i]\n            nom = td.timeseries[w]["NOMINAL"]\n            x = np.arange(len(nom), dtype=float) / 60.0  # minutes (1Hz grid)\n\n            stack = np.vstack([td.timeseries[w][a] for a in ANOMALIES])\n            anom_mean = np.mean(stack, axis=0)\n            anom_min = np.min(stack, axis=0)\n            anom_max = np.max(stack, axis=0)\n\n            ax.plot(x, nom, color="black", linewidth=2.4, label="Benign (NOMINAL)")\n            for a in ANOMALIES:\n                ax.plot(\n                    x,\n                    td.timeseries[w][a],\n                    color=COLOR_BY_STRESSOR[a],\n                    alpha=0.6,\n                    linewidth=1.0,\n                    label=a,\n                )\n            ax.plot(x, anom_mean, color="#c62828", linewidth=2.2, label="Anomaly mean")\n            ax.fill_between(x, anom_min, anom_max, color="#ef5350", alpha=0.18, label="Anomaly range")\n            ax.set_ylabel("AF Index", fontsize=14)\n            ax.set_xlabel("Time (minutes)", fontsize=14)\n            ax.set_title(w, fontsize=16, fontweight="bold")\n            ax.grid(alpha=0.25)\n            ax.tick_params(axis="both", labelsize=12)\n\n        h, l = axes[0].get_legend_handles_labels()\n        dedup = dict(zip(l, h))\n        fig.legend(\n            dedup.values(),\n            dedup.keys(),\n            loc="upper center",\n            ncol=4,\n            frameon=True,\n            fontsize=12,\n            bbox_to_anchor=(0.5, 1.02),\n        )\n        fig.suptitle(f"All-feature AF-index trajectories | {TIER_PRETTY[td.tier]}", fontsize=20, y=1.04)\n        fig.tight_layout(rect=[0, 0, 1, 0.98])\n\n        out = out_dir / f"fig_af_timeseries_{td.tier}.png"\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_auc_heatmaps(out_dir: Path, stressor: pd.DataFrame) -> List[str]:\n    out_paths = []\n    for metric, title, fname in [\n        ("roc_auc", "ROC-AUC by tier and stressor", "fig_heatmap_roc_auc.png"),\n        ("pr_auc", "AUC-PR by tier and stressor", "fig_heatmap_pr_auc.png"),\n    ]:\n        piv = stressor.pivot(index="stressor", columns="tier_name", values=metric).loc[ANOMALIES]\n        cols = [c for c in ["Tier-0", "Tier-1", "Tier-2"] if c in piv.columns]\n        piv = piv[cols]\n\n        fig, ax = plt.subplots(figsize=(8.5, 4.5))\n        im = ax.imshow(piv.to_numpy(dtype=float), vmin=0.5, vmax=1.0, cmap="viridis")\n        ax.set_xticks(np.arange(len(piv.columns)))\n        ax.set_xticklabels(piv.columns, fontsize=12)\n        ax.set_yticks(np.arange(len(piv.index)))\n        ax.set_yticklabels(piv.index, fontsize=12)\n        ax.set_title(title, fontsize=16, fontweight="bold")\n        for i in range(len(piv.index)):\n            for j in range(len(piv.columns)):\n                v = float(piv.iloc[i, j])\n                ax.text(j, i, f"{v:.3f}", ha="center", va="center", color="white", fontsize=11)\n        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n        cbar.ax.set_ylabel(metric.upper(), rotation=90, fontsize=11)\n        fig.tight_layout()\n        out = out_dir / fname\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_run_score_distributions(out_dir: Path, runs: pd.DataFrame) -> str:\n    tiers = ["tier0", "tier1_alt", "tier2"]\n    fig, axes = plt.subplots(1, len(tiers), figsize=(14.5, 4.6), sharey=False)\n    if len(tiers) == 1:\n        axes = [axes]\n\n    for i, t in enumerate(tiers):\n        ax = axes[i]\n        d = runs[runs["tier"] == t]\n        nom = d[d["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = d[d["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        bp = ax.boxplot([nom, anm], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(nom)), nom, color="black", s=24, alpha=0.8)\n        ax.scatter(np.repeat(2, len(anm)), anm, color="#c62828", s=24, alpha=0.7)\n        ax.set_title(TIER_PRETTY[t], fontsize=14, fontweight="bold")\n        ax.set_ylabel("Run AF Index (median)", fontsize=12)\n        ax.grid(alpha=0.22)\n        ax.tick_params(axis="both", labelsize=11)\n\n    fig.suptitle("Run-level AF-index score distributions", fontsize=18, y=1.02)\n    fig.tight_layout()\n    out = out_dir / "fig_run_score_distributions.png"\n    fig.savefig(out, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n    return str(out)\n\n\ndef build_feature_inventory(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": len(td.features),\n                "n_features_union": len(td.union_features),\n                "common_features_json": json.dumps(td.features),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_markdown_summary(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    fig_paths: List[str],\n) -> None:\n    best_tier = overall.sort_values("pr_auc", ascending=False).iloc[0]\n    weakest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=True)\n        .head(2)\n        .index.tolist()\n    )\n    strongest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=False)\n        .head(3)\n        .index.tolist()\n    )\n    lines = []\n    lines.append("# DICE Results/Analysis Auto-Summary")\n    lines.append("")\n    lines.append("## Key Findings")\n    lines.append(\n        f"- Best run-level AUC-PR tier: **{best_tier[\'tier_name\']}** "\n        f"(AUC-PR={best_tier[\'pr_auc\']:.4f}, ROC-AUC={best_tier[\'roc_auc\']:.4f})."\n    )\n    lines.append(f"- Strongest stressors (mean AUC-PR across tiers): **{\', \'.join(strongest)}**.")\n    lines.append(f"- Hardest stressors (mean AUC-PR across tiers): **{\', \'.join(weakest)}**.")\n    lines.append("")\n    lines.append("## Suggested Results Narrative")\n    lines.append(\n        "Across the 24-run Apple dataset, AF-index separation is consistently visible between nominal and "\n        "anomalous runs in all telemetry tiers. Tier-aware scoring indicates that anomaly/nominal score ratios "\n        "remain above 1.0 in every tier, confirming stable separability under the fixed collection protocol. "\n        "Per-stressor analysis shows stronger separation for ATOMIC, CACHE, and MEMBW, while BRANCH and TLB "\n        "remain comparatively harder due to weaker host-visible signatures. These observations match the "\n        "expected mechanism-level difficulty ordering in software-driven stressors."\n    )\n    lines.append("")\n    lines.append("## Generated Figures")\n    for p in fig_paths:\n        lines.append(f"- `{p}`")\n    lines.append("")\n    lines.append("## Generated Tables")\n    for p in [\n        out_dir / "table_overall_metrics.csv",\n        out_dir / "table_stressor_metrics.csv",\n        out_dir / "table_workload_summary.csv",\n        out_dir / "table_feature_inventory.csv",\n        out_dir / "table_overall_metrics.tex",\n        out_dir / "table_stressor_metrics.tex",\n    ]:\n        lines.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(lines) + "\\n")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n        help="Dataset root containing tier0, tier1_alt, tier2 folders.",\n    )\n    ap.add_argument(\n        "--out_dir",\n        type=Path,\n        default=None,\n        help="Output directory for results tables/figures (default: <root>/results_analysis).",\n    )\n    ap.add_argument("--source_hz", type=int, default=5, help="Source sampling Hz used for downsampling.")\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = (args.out_dir.expanduser().resolve() if args.out_dir else (root / "results_analysis"))\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir = out_dir / "figures"\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_data = [build_tier_data(root, t, source_hz=args.source_hz) for t in ["tier0", "tier1_alt", "tier2"]]\n\n    runs = pd.concat([td.run_df for td in tier_data], ignore_index=True)\n    quality = pd.concat([td.case_quality for td in tier_data], ignore_index=True)\n    features = build_feature_inventory(tier_data)\n    overall = make_overall_metrics(tier_data)\n    stressor = make_stressor_metrics(tier_data)\n    workload = make_workload_summary(tier_data)\n\n    save_metric_tables(out_dir, overall, stressor, workload, features, quality, runs)\n\n    figs = []\n    figs.extend(plot_af_timeseries(fig_dir, tier_data))\n    figs.extend(plot_auc_heatmaps(fig_dir, stressor))\n    figs.append(plot_run_score_distributions(fig_dir, runs))\n\n    write_markdown_summary(out_dir, overall, stressor, figs)\n\n    print(f"[OK] Results generated at: {out_dir}")\n    print("[OK] Figures:")\n    for p in figs:\n        print(f" - {p}")\n    print("[OK] Tables:")\n    print(f" - {out_dir / \'table_overall_metrics.csv\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.csv\'}")\n    print(f" - {out_dir / \'table_workload_summary.csv\'}")\n    print(f" - {out_dir / \'table_overall_metrics.tex\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.tex\'}")\n\n\nif __name__ == "__main__":\n    main()\n'
NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""\nFull retrain/evaluation for DICE micro-twin + split-conformal pipeline.\n\nProtocol:\n- Use Tier-0 / Tier-1-alt / Tier-2 clean dataset (5000 rows @ 5Hz per run).\n- Align to 1Hz via mean pooling.\n- Train only on benign runs (NOMINAL) with workload-holdout folds.\n- Fit linear micro-twin dynamics in normalized feature space.\n- Build residual signatures on decision blocks.\n- Calibrate conformal threshold on benign calibration blocks.\n- Evaluate run-level labels (Benign vs Anomaly) via persistent block alerts.\n\nOutputs:\n- CSV metrics tables and per-case predictions\n- LaTeX table snippets for paper\n- ROC/PR and score distribution figures\n- Markdown summary for direct paste into Results section\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Sequence, Tuple\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    roc_auc_score,\n    roc_curve,\n)\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nIGNORE_COLS = {"idx", "ts_unix_s", "t_rel_s", "timestamp", "time", "ts"}\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nCONFIGS = {\n    "tier0": ["tier0"],\n    "tier0_tier1": ["tier0", "tier1_alt"],\n    "tier0_tier1_tier2": ["tier0", "tier1_alt", "tier2"],\n}\n\nDIAG_TOP_K = 5\nMECHANISM_GROUPS = [\n    "compute",\n    "memory_io",\n    "thermal_power",\n    "scheduler_runtime",\n    "platform_pressure",\n]\n\n\n@dataclass(frozen=True)\nclass CaseRef:\n    workload: str\n    stressor: str\n\n    @property\n    def case_id(self) -> str:\n        return f"{self.workload}__{self.stressor}"\n\n    @property\n    def label(self) -> int:\n        return 0 if self.stressor == "NOMINAL" else 1\n\n\n@dataclass\nclass ModelBundle:\n    feature_names: List[str]\n    median: np.ndarray\n    scale: np.ndarray\n    A: np.ndarray\n    weights: np.ndarray\n    cal_scores: np.ndarray\n    tau: float\n\n\ndef all_cases() -> List[CaseRef]:\n    return [CaseRef(w, s) for w in WORKLOADS for s in STRESSORS]\n\n\ndef robust_scale_1d(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef robust_fit_matrix(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.nanmedian(X, axis=0)\n    scale = np.zeros(X.shape[1], dtype=float)\n    for j in range(X.shape[1]):\n        scale[j] = robust_scale_1d(X[:, j])\n    scale[scale <= 1e-12] = 1.0\n    return med, scale\n\n\ndef safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, score))\n\n\ndef safe_ap(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, score))\n\n\ndef case_path(root: Path, tier: str, case: CaseRef) -> Path:\n    return root / tier / case.case_id / TIER_FILE[tier]\n\n\ndef read_df(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing file: {path}")\n    df = pd.read_csv(path)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_features(df: pd.DataFrame) -> List[str]:\n    out = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            out.append(c)\n    return out\n\n\ndef downsample_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    tmp = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return tmp.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef common_features_per_tier(root: Path, tier: str) -> List[str]:\n    common = None\n    for case in all_cases():\n        df = read_df(case_path(root, tier, case))\n        cols = set(numeric_features(df))\n        common = cols if common is None else (common & cols)\n    common_list = sorted(common) if common else []\n\n    # Drop globally constant features.\n    keep = []\n    for f in common_list:\n        vals = []\n        for case in all_cases():\n            d = downsample_1hz(read_df(case_path(root, tier, case)))\n            vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep\n\n\ndef build_case_matrix(\n    root: Path,\n    case: CaseRef,\n    tiers: Sequence[str],\n    feature_map: Dict[str, List[str]],\n    source_hz: int = 5,\n) -> Tuple[np.ndarray, List[str]]:\n    mats = []\n    names = []\n    lengths = []\n    for t in tiers:\n        df = downsample_1hz(read_df(case_path(root, t, case)), source_hz=source_hz)\n        feats = feature_map[t]\n        arr = df[feats].to_numpy(dtype=float)\n        mats.append(arr)\n        lengths.append(arr.shape[0])\n        names.extend([f"{t}:{f}" for f in feats])\n\n    n = min(lengths)\n    mats = [m[:n] for m in mats]\n    X = np.concatenate(mats, axis=1)\n    return X, names\n\n\ndef fit_linear_dynamics(X_runs: List[np.ndarray], ridge_lambda: float = 1e-3) -> np.ndarray:\n    X_prev = []\n    X_next = []\n    for X in X_runs:\n        if len(X) < 2:\n            continue\n        X_prev.append(X[:-1])\n        X_next.append(X[1:])\n    if not X_prev:\n        raise RuntimeError("Not enough samples to fit dynamics.")\n    P = np.vstack(X_prev)  # [N, d]\n    N = np.vstack(X_next)  # [N, d]\n    d = P.shape[1]\n    xtx = P.T @ P + ridge_lambda * np.eye(d)\n    xty = P.T @ N\n    A = np.linalg.solve(xtx, xty)  # [d, d]\n    return A\n\n\ndef residual_timeseries(X_norm: np.ndarray, A: np.ndarray, gain: float) -> np.ndarray:\n    """\n    Kalman-style fixed-gain synchronization:\n    z_pred = A z_prev\n    r_t    = x_t - z_pred\n    z_t    = z_pred + gain * r_t\n    """\n    T, d = X_norm.shape\n    if T < 2:\n        return np.zeros((0, d), dtype=float)\n    z = X_norm[0].copy()\n    residuals = []\n    for t in range(1, T):\n        z_pred = z @ A\n        r = X_norm[t] - z_pred\n        residuals.append(r)\n        z = z_pred + gain * r\n    return np.vstack(residuals)\n\n\ndef block_signatures(residual: np.ndarray, B: int) -> np.ndarray:\n    """\n    Signature per block: mean absolute residual over a sliding window.\n    """\n    if residual.shape[0] == 0:\n        return np.zeros((0, residual.shape[1]), dtype=float)\n    a = np.abs(residual)\n    T, d = a.shape\n    if T < B:\n        return np.mean(a, axis=0, keepdims=True)\n    cs = np.vstack([np.zeros((1, d)), np.cumsum(a, axis=0)])\n    out = (cs[B:] - cs[:-B]) / float(B)\n    return out\n\n\ndef fit_weights(signatures_fit: np.ndarray) -> np.ndarray:\n    sigma = np.std(signatures_fit, axis=0)\n    w = 1.0 / (sigma + 1e-6)\n    w = np.maximum(w, 0.0)\n    s = np.sum(w)\n    if s <= 0:\n        return np.ones_like(w) / len(w)\n    return w / s\n\n\ndef signature_scores(signatures: np.ndarray, weights: np.ndarray) -> np.ndarray:\n    if signatures.shape[0] == 0:\n        return np.zeros((0,), dtype=float)\n    return signatures @ weights\n\n\ndef conformal_threshold(cal_scores: np.ndarray, alpha: float) -> float:\n    sc = np.sort(np.asarray(cal_scores, dtype=float))\n    n = len(sc)\n    if n == 0:\n        return float("inf")\n    k = int(np.ceil((n + 1) * (1.0 - alpha)))\n    k = min(max(k, 1), n)\n    return float(sc[k - 1])\n\n\ndef conformal_pvals(cal_scores: np.ndarray, test_scores: np.ndarray) -> np.ndarray:\n    cal = np.asarray(cal_scores, dtype=float)\n    denom = len(cal) + 1.0\n    out = np.zeros(len(test_scores), dtype=float)\n    for i, s in enumerate(test_scores):\n        out[i] = (1.0 + np.sum(cal >= s)) / denom\n    return out\n\n\ndef persistent_alerts(alerts: np.ndarray, k: int) -> np.ndarray:\n    out = np.zeros(len(alerts), dtype=int)\n    run = 0\n    for i, a in enumerate(alerts.astype(bool)):\n        if a:\n            run += 1\n        else:\n            run = 0\n        out[i] = 1 if run >= k else 0\n    return out\n\n\ndef first_positive_index(x: np.ndarray) -> int:\n    idx = np.flatnonzero(np.asarray(x, dtype=bool))\n    return int(idx[0]) if len(idx) else -1\n\n\ndef finite_median(x: Sequence[float]) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.median(arr))\n\n\ndef finite_percentile(x: Sequence[float], q: float) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.percentile(arr, q))\n\n\ndef mechanism_group(feature_name: str) -> str:\n    name = feature_name.split(":", 1)[-1].lower()\n    if any(tok in name for tok in ["temp", "power", "fan"]):\n        return "thermal_power"\n    if any(tok in name for tok in ["mem_", "swap_", "disk_", "net_", "wired_bytes", "active_bytes", "inactive_bytes"]):\n        return "memory_io"\n    if any(\n        tok in name\n        for tok in [\n            "ctx_switch",\n            "interrupt",\n            "syscall",\n            "pids_count",\n            "running_fraction",\n            "weight_ns",\n            "unique_process",\n            "unique_thread",\n            "samples_per_bucket",\n            "sentinel_count",\n            "core_id",\n        ]\n    ):\n        return "scheduler_runtime"\n    if any(tok in name for tok in ["load", "uptime", "available_bytes", "free_bytes", "mem_percent"]):\n        return "platform_pressure"\n    return "compute"\n\n\ndef mechanism_vector(feature_names: Sequence[str], feature_contrib: np.ndarray) -> Tuple[Dict[str, float], np.ndarray]:\n    totals = {group: 0.0 for group in MECHANISM_GROUPS}\n    for name, value in zip(feature_names, np.asarray(feature_contrib, dtype=float)):\n        totals[mechanism_group(name)] += float(value)\n    vec = np.array([totals[group] for group in MECHANISM_GROUPS], dtype=float)\n    return totals, vec\n\n\ndef train_bundle(\n    train_benign_runs: Dict[str, np.ndarray],\n    feature_names: List[str],\n    fit_ratio: float,\n    B: int,\n    alpha: float,\n    gain: float,\n    ridge_lambda: float,\n) -> ModelBundle:\n    fit_runs = []\n    cal_runs = []\n    fit_samples = []\n\n    for _, X in train_benign_runs.items():\n        n = len(X)\n        split = int(max(2, min(n - 1, round(n * fit_ratio))))\n        X_fit = X[:split]\n        X_cal = X[split:]\n        fit_runs.append(X_fit)\n        cal_runs.append(X_cal if len(X_cal) > 1 else X_fit[-2:])\n        fit_samples.append(X_fit)\n\n    X_fit_all = np.vstack(fit_samples)\n    med, scale = robust_fit_matrix(X_fit_all)\n\n    fit_norm = [(x - med) / (scale + 1e-12) for x in fit_runs]\n    cal_norm = [(x - med) / (scale + 1e-12) for x in cal_runs]\n\n    A = fit_linear_dynamics(fit_norm, ridge_lambda=ridge_lambda)\n\n    sig_fit = []\n    for X in fit_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        if len(s):\n            sig_fit.append(s)\n    sig_fit_all = np.vstack(sig_fit)\n    w = fit_weights(sig_fit_all)\n\n    cal_scores = []\n    for X in cal_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        sc = signature_scores(s, w)\n        if len(sc):\n            cal_scores.append(sc)\n    cal_scores_all = np.concatenate(cal_scores)\n    tau = conformal_threshold(cal_scores_all, alpha=alpha)\n\n    return ModelBundle(\n        feature_names=feature_names,\n        median=med,\n        scale=scale,\n        A=A,\n        weights=w,\n        cal_scores=cal_scores_all,\n        tau=tau,\n    )\n\n\ndef evaluate_run(\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> Tuple[Dict[str, float], np.ndarray]:\n    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)\n    r = residual_timeseries(Xn, bundle.A, gain=gain)\n    sig = block_signatures(r, B=B)\n    sc = signature_scores(sig, bundle.weights)\n    pv = conformal_pvals(bundle.cal_scores, sc)\n\n    block_alert = pv < alpha\n    persist = persistent_alerts(block_alert, k=persist_k)\n\n    run_alert = int(np.any(persist > 0))\n    peak_score = float(np.max(sc)) if len(sc) else 0.0\n    run_score = peak_score\n    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)\n    feature_contrib = run_signature * bundle.weights\n    first_block_idx = first_positive_index(block_alert)\n    first_persist_idx = first_positive_index(persist)\n    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")\n    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")\n    n_blocks = int(len(sc))\n    duration_s = max(n_blocks, 1)\n\n    return (\n        {\n            "run_score": run_score,\n            "run_alert": run_alert,\n            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,\n            "peak_block_score": peak_score,\n            "n_blocks": n_blocks,\n            "n_block_alerts": int(np.sum(block_alert)),\n            "n_persist_alerts": int(np.sum(persist)),\n            "first_block_alert_idx": first_block_idx,\n            "first_persist_alert_idx": first_persist_idx,\n            "first_block_alert_s": block_time_s,\n            "time_to_detect_s": persist_time_s,\n            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),\n            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),\n        },\n        feature_contrib,\n    )\n\n\ndef to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:\n    body = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{body}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef plot_curves(df: pd.DataFrame, out_png: Path, score_col: str, title_tag: str) -> None:\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    for cfg, d in df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s = d[score_col].to_numpy(dtype=float)\n        if len(np.unique(y)) < 2:\n            continue\n        fpr, tpr, _ = roc_curve(y, s)\n        p, r, _ = precision_recall_curve(y, s)\n        axes[0].plot(fpr, tpr, linewidth=2, label=f"{cfg} (AUC={roc_auc_score(y, s):.3f})")\n        axes[1].plot(r, p, linewidth=2, label=f"{cfg} (AP={average_precision_score(y, s):.3f})")\n    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)\n    axes[0].set_title("ROC curve")\n    axes[0].set_xlabel("False Positive Rate")\n    axes[0].set_ylabel("True Positive Rate")\n    axes[1].set_title("Precision-Recall curve")\n    axes[1].set_xlabel("Recall")\n    axes[1].set_ylabel("Precision")\n    for ax in axes:\n        ax.grid(alpha=0.25)\n        ax.legend(frameon=True, fontsize=10)\n    fig.suptitle(f"DICE curves ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_score_box(df: pd.DataFrame, out_png: Path, score_col: str, y_label: str, title_tag: str) -> None:\n    cfgs = list(df["config"].unique())\n    fig, axes = plt.subplots(1, len(cfgs), figsize=(5.0 * len(cfgs), 4.8), sharey=False)\n    if len(cfgs) == 1:\n        axes = [axes]\n    for i, cfg in enumerate(cfgs):\n        ax = axes[i]\n        d = df[df["config"] == cfg]\n        neg = d[d["label"] == 0][score_col].to_numpy(dtype=float)\n        pos = d[d["label"] == 1][score_col].to_numpy(dtype=float)\n        bp = ax.boxplot([neg, pos], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(neg)), neg, color="black", s=22, alpha=0.8)\n        ax.scatter(np.repeat(2, len(pos)), pos, color="#c62828", s=22, alpha=0.7)\n        ax.set_title(cfg)\n        ax.set_ylabel(y_label)\n        ax.grid(alpha=0.22)\n    fig.suptitle(f"DICE run score distributions ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef build_diagnostic_record(\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    feature_names: Sequence[str],\n    feature_contrib: np.ndarray,\n) -> Dict[str, object]:\n    contrib = np.asarray(feature_contrib, dtype=float)\n    total = float(np.sum(contrib))\n    tier_totals = {tier: 0.0 for tier in TIER_FILE}\n    for name, value in zip(feature_names, contrib):\n        tier = name.split(":", 1)[0]\n        if tier in tier_totals:\n            tier_totals[tier] += float(value)\n    dominant_tier = max(tier_totals, key=tier_totals.get) if total > 0 else "none"\n    mech_totals, mech_vec = mechanism_vector(feature_names, contrib)\n    dominant_mechanism = max(mech_totals, key=mech_totals.get) if total > 0 else "none"\n    order = np.argsort(contrib)[::-1][:DIAG_TOP_K]\n    mech_order = np.argsort(mech_vec)[::-1][:3]\n\n    row: Dict[str, object] = {\n        "config": config,\n        "holdout_workload": holdout_workload,\n        "case_id": case.case_id,\n        "workload": case.workload,\n        "stressor": case.stressor,\n        "label": case.label,\n        "dominant_tier": dominant_tier,\n        "tier0_contrib": float(tier_totals["tier0"]),\n        "tier1_alt_contrib": float(tier_totals["tier1_alt"]),\n        "tier2_contrib": float(tier_totals["tier2"]),\n        "tier0_share": float(tier_totals["tier0"] / total) if total > 0 else 0.0,\n        "tier1_alt_share": float(tier_totals["tier1_alt"] / total) if total > 0 else 0.0,\n        "tier2_share": float(tier_totals["tier2"] / total) if total > 0 else 0.0,\n        "dominant_mechanism": dominant_mechanism,\n        "_feature_contrib": contrib.copy(),\n        "_mechanism_vector": mech_vec.copy(),\n    }\n    for group in MECHANISM_GROUPS:\n        row[f"{group}_contrib"] = float(mech_totals[group])\n        row[f"{group}_share"] = float(mech_totals[group] / total) if total > 0 else 0.0\n    for rank in range(DIAG_TOP_K):\n        key_name = f"top_feature_{rank + 1}"\n        key_score = f"top_feature_score_{rank + 1}"\n        if rank < len(order) and contrib[order[rank]] > 0.0:\n            idx = int(order[rank])\n            row[key_name] = feature_names[idx]\n            row[key_score] = float(contrib[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    for rank in range(3):\n        key_name = f"top_mechanism_{rank + 1}"\n        key_score = f"top_mechanism_score_{rank + 1}"\n        if rank < len(mech_order) and mech_vec[mech_order[rank]] > 0.0:\n            idx = int(mech_order[rank])\n            row[key_name] = MECHANISM_GROUPS[idx]\n            row[key_score] = float(mech_vec[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    return row\n\n\ndef append_case_outputs(\n    preds: List[Dict[str, object]],\n    diagnostic_records: List[Dict[str, object]],\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> None:\n    metrics, feature_contrib = evaluate_run(\n        X_run,\n        bundle,\n        B=B,\n        alpha=alpha,\n        persist_k=persist_k,\n        gain=gain,\n    )\n    preds.append(\n        {\n            "config": config,\n            "holdout_workload": holdout_workload,\n            "case_id": case.case_id,\n            "workload": case.workload,\n            "stressor": case.stressor,\n            "label": case.label,\n            **metrics,\n            "n_features": len(bundle.feature_names),\n            "tau": bundle.tau,\n        }\n    )\n    diagnostic_records.append(\n        build_diagnostic_record(\n            config=config,\n            holdout_workload=holdout_workload,\n            case=case,\n            feature_names=bundle.feature_names,\n            feature_contrib=feature_contrib,\n        )\n    )\n\n\ndef build_stressor_attribution(\n    diagnostic_records: Sequence[Dict[str, object]],\n    vector_key: str,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pred_rows: List[Dict[str, object]] = []\n    for cfg_name in CONFIGS:\n        cfg_records = [r for r in diagnostic_records if r["config"] == cfg_name and int(r["label"]) == 1]\n        for holdout_w in WORKLOADS:\n            train = [r for r in cfg_records if r["workload"] != holdout_w]\n            test = [r for r in cfg_records if r["workload"] == holdout_w]\n            centroids = {}\n            for stressor in ANOMALIES:\n                mats = [r[vector_key] for r in train if r["stressor"] == stressor]\n                if mats:\n                    centroids[stressor] = np.median(np.vstack(mats), axis=0)\n            if len(centroids) < 2:\n                continue\n            for row in test:\n                truth = str(row["stressor"])\n                contrib = np.asarray(row[vector_key], dtype=float)\n                dists = {stressor: float(np.linalg.norm(contrib - centroid)) for stressor, centroid in centroids.items()}\n                ordered = sorted(dists.items(), key=lambda item: item[1])\n                pred = ordered[0][0]\n                top2 = [label for label, _ in ordered[:2]]\n                pred_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "case_id": row["case_id"],\n                        "true_stressor": truth,\n                        "pred_stressor": pred,\n                        "is_correct": int(pred == truth),\n                        "top2_hit": int(truth in top2),\n                        "nearest_distance": float(ordered[0][1]),\n                        "margin_to_second": float(ordered[1][1] - ordered[0][1]) if len(ordered) > 1 else float("inf"),\n                    }\n                )\n\n    pred_df = pd.DataFrame(pred_rows)\n    if pred_df.empty:\n        empty_metrics = pd.DataFrame(\n            columns=["config", "n_cases", "top1_acc", "top2_acc", "macro_f1", "mean_margin_to_second"]\n        )\n        empty_cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n        empty_cm.index.name = "true_stressor"\n        empty_cm.columns.name = "pred_stressor"\n        return pred_df, empty_metrics, empty_cm\n    pred_df = pred_df.sort_values(["config", "holdout_workload", "case_id"])\n\n    metric_rows = []\n    for cfg_name, d in pred_df.groupby("config", sort=False):\n        metric_rows.append(\n            {\n                "config": cfg_name,\n                "n_cases": int(len(d)),\n                "top1_acc": float(d["is_correct"].mean()),\n                "top2_acc": float(d["top2_hit"].mean()),\n                "macro_f1": float(\n                    f1_score(\n                        d["true_stressor"],\n                        d["pred_stressor"],\n                        labels=ANOMALIES,\n                        average="macro",\n                        zero_division=0,\n                    )\n                ),\n                "mean_margin_to_second": float(d["margin_to_second"].replace([np.inf, -np.inf], np.nan).mean()),\n            }\n        )\n    metrics_df = pd.DataFrame(metric_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    d_final = pred_df[pred_df["config"] == final_cfg]\n    if d_final.empty:\n        cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n    else:\n        cm_arr = confusion_matrix(\n            d_final["true_stressor"],\n            d_final["pred_stressor"],\n            labels=ANOMALIES,\n        )\n        cm = pd.DataFrame(cm_arr, index=ANOMALIES, columns=ANOMALIES)\n    cm.index.name = "true_stressor"\n    cm.columns.name = "pred_stressor"\n    return pred_df, metrics_df, cm\n\n\ndef build_stressor_tier_contributions(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    cols = ["tier0_share", "tier1_alt_share", "tier2_share"]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *cols, "dominant_tier_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_tier"].mode()\n        rows.append(\n            {\n                "stressor": stressor,\n                "tier0_share": float(part["tier0_share"].mean()),\n                "tier1_alt_share": float(part["tier1_alt_share"].mean()),\n                "tier2_share": float(part["tier2_share"].mean()),\n                "dominant_tier_mode": str(mode.iloc[0]) if not mode.empty else "none",\n            }\n        )\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_mechanism_summary(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    share_cols = [f"{group}_share" for group in MECHANISM_GROUPS]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *share_cols, "dominant_mechanism_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_mechanism"].mode()\n        row = {\n            "stressor": stressor,\n            "dominant_mechanism_mode": str(mode.iloc[0]) if not mode.empty else "none",\n        }\n        for col in share_cols:\n            row[col] = float(part[col].mean())\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_sequential_metrics(pred_df: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for cfg, d in pred_df.groupby("config", sort=False):\n        benign = d[d["label"] == 0]\n        anomaly = d[d["label"] == 1]\n        detected = anomaly[anomaly["run_alert"] == 1]\n        rows.append(\n            {\n                "config": cfg,\n                "benign_run_alert_rate": float(benign["run_alert"].mean()),\n                "benign_persist_alerts_per_hour": float(benign["persist_alerts_per_hour"].mean()),\n                "benign_block_alerts_per_hour": float(benign["block_alerts_per_hour"].mean()),\n                "anomaly_detect_rate": float(anomaly["run_alert"].mean()),\n                "median_time_to_detect_s": finite_median(detected["time_to_detect_s"]),\n                "p90_time_to_detect_s": finite_percentile(detected["time_to_detect_s"], 90),\n                "detect_within_120s": float((anomaly["time_to_detect_s"] <= 120).fillna(False).mean()),\n                "detect_within_300s": float((anomaly["time_to_detect_s"] <= 300).fillna(False).mean()),\n                "detect_within_600s": float((anomaly["time_to_detect_s"] <= 600).fillna(False).mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef build_holdout_robustness_summary(fold_df: pd.DataFrame) -> pd.DataFrame:\n    d = fold_df[fold_df["holdout_workload"] != "ALL"].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["config", "mean_pr_auc", "worst_pr_auc", "mean_roc_auc", "mean_fpr", "mean_tpr"])\n    rows = []\n    for cfg, part in d.groupby("config", sort=False):\n        rows.append(\n            {\n                "config": cfg,\n                "mean_pr_auc": float(part["pr_auc"].mean()),\n                "worst_pr_auc": float(part["pr_auc"].min()),\n                "mean_roc_auc": float(part["roc_auc"].mean()),\n                "mean_fpr": float(part["fpr"].mean()),\n                "mean_tpr": float(part["tpr"].mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef plot_confusion_heatmap(cm: pd.DataFrame, out_png: Path, title: str) -> None:\n    if cm.empty:\n        return\n    mat = cm.to_numpy(dtype=float)\n    fig, ax = plt.subplots(figsize=(6.2, 5.2))\n    im = ax.imshow(mat, cmap="Blues")\n    ax.set_xticks(np.arange(len(cm.columns)), labels=list(cm.columns), rotation=30, ha="right")\n    ax.set_yticks(np.arange(len(cm.index)), labels=list(cm.index))\n    ax.set_xlabel("Predicted stressor")\n    ax.set_ylabel("True stressor")\n    ax.set_title(title)\n    for i in range(mat.shape[0]):\n        for j in range(mat.shape[1]):\n            color = "white" if mat[i, j] >= max(1.0, np.max(mat) * 0.55) else "black"\n            ax.text(j, i, f"{int(mat[i, j])}", ha="center", va="center", color=color, fontsize=10)\n    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_stressor_tier_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.4, 4.8))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("tier0_share", "Tier-0", "#78909c"),\n        ("tier1_alt_share", "Tier-1", "#81c784"),\n        ("tier2_share", "Tier-2", "#ffb74d"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean contribution share")\n    ax.set_title("Final-config diagnosis contribution share by tier")\n    ax.legend(frameon=True)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_mechanism_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(8.6, 5.0))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("compute_share", "Compute", "#5c6bc0"),\n        ("memory_io_share", "Memory/I/O", "#26a69a"),\n        ("thermal_power_share", "Thermal/Power", "#ef5350"),\n        ("scheduler_runtime_share", "Scheduler/Runtime", "#8d6e63"),\n        ("platform_pressure_share", "Platform Pressure", "#78909c"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean mechanism share")\n    ax.set_title("Final-config mechanism diagnosis share by stressor")\n    ax.legend(frameon=True, ncol=2)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_detection_latency(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.0, 4.8))\n    vals = df["median_time_to_detect_s"].to_numpy(dtype=float)\n    ax.bar(df["config"], vals, color=["#90a4ae", "#66bb6a", "#ffa726"][: len(df)])\n    ax.set_ylabel("Median time-to-detect (s)")\n    ax.set_title("Sequential detection latency by observation head")\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef append_text_table(lines: List[str], df: pd.DataFrame) -> None:\n    lines.append("```text")\n    lines.append(df.to_string(index=False))\n    lines.append("```")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n    )\n    ap.add_argument("--out_dir", type=Path, default=None)\n    ap.add_argument("--source_hz", type=int, default=5)\n    ap.add_argument("--fit_ratio", type=float, default=0.6)\n    ap.add_argument("--block_B", type=int, default=60)\n    ap.add_argument("--alpha", type=float, default=0.05)\n    ap.add_argument("--persist_k", type=int, default=3)\n    ap.add_argument("--gain", type=float, default=0.35)\n    ap.add_argument("--ridge_lambda", type=float, default=1e-3)\n    ap.add_argument(\n        "--protocol",\n        choices=["workload_holdout", "global"],\n        default="global",\n        help="Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).",\n    )\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = args.out_dir.expanduser().resolve() if args.out_dir else root / "results_dice_full"\n    fig_dir = out_dir / "figures"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_features = {t: common_features_per_tier(root, t) for t in TIER_FILE.keys()}\n    for t, fs in tier_features.items():\n        print(f"[INFO] {t}: common features={len(fs)}")\n\n    preds = []\n    fold_rows = []\n    diagnostic_records: List[Dict[str, object]] = []\n\n    for cfg_name, tiers in CONFIGS.items():\n        print(f"[INFO] training config={cfg_name} tiers={tiers}")\n        case_X = {}\n        feature_names_cfg = None\n        for case in all_cases():\n            X, names = build_case_matrix(\n                root,\n                case,\n                tiers=tiers,\n                feature_map=tier_features,\n                source_hz=args.source_hz,\n            )\n            case_X[case.case_id] = X\n            if feature_names_cfg is None:\n                feature_names_cfg = names\n\n        if args.protocol == "workload_holdout":\n            for holdout_w in WORKLOADS:\n                train_benign = {\n                    case_id: X\n                    for case_id, X in case_X.items()\n                    if case_id.endswith("__NOMINAL") and not case_id.startswith(f"{holdout_w}__")\n                }\n\n                bundle = train_bundle(\n                    train_benign_runs=train_benign,\n                    feature_names=feature_names_cfg or [],\n                    fit_ratio=args.fit_ratio,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    gain=args.gain,\n                    ridge_lambda=args.ridge_lambda,\n                )\n\n                test_cases = [c for c in all_cases() if c.workload == holdout_w]\n                for case in test_cases:\n                    append_case_outputs(\n                        preds=preds,\n                        diagnostic_records=diagnostic_records,\n                        config=cfg_name,\n                        holdout_workload=holdout_w,\n                        case=case,\n                        X_run=case_X[case.case_id],\n                        bundle=bundle,\n                        B=args.block_B,\n                        alpha=args.alpha,\n                        persist_k=args.persist_k,\n                        gain=args.gain,\n                    )\n\n                fold_curr = [p for p in preds if p["config"] == cfg_name and p["holdout_workload"] == holdout_w]\n                fd = pd.DataFrame(fold_curr)\n                y = fd["label"].to_numpy(dtype=int)\n                s_run = fd["run_score"].to_numpy(dtype=float)\n                fold_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "roc_auc": safe_auc(y, s_run),\n                        "pr_auc": safe_ap(y, s_run),\n                        "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                        "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                        "n_features": int(fd["n_features"].iloc[0]),\n                    }\n                )\n        else:\n            train_benign = {case_id: X for case_id, X in case_X.items() if case_id.endswith("__NOMINAL")}\n            bundle = train_bundle(\n                train_benign_runs=train_benign,\n                feature_names=feature_names_cfg or [],\n                fit_ratio=args.fit_ratio,\n                B=args.block_B,\n                alpha=args.alpha,\n                gain=args.gain,\n                ridge_lambda=args.ridge_lambda,\n            )\n            for case in all_cases():\n                append_case_outputs(\n                    preds=preds,\n                    diagnostic_records=diagnostic_records,\n                    config=cfg_name,\n                    holdout_workload="ALL",\n                    case=case,\n                    X_run=case_X[case.case_id],\n                    bundle=bundle,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    persist_k=args.persist_k,\n                    gain=args.gain,\n                )\n\n            fd = pd.DataFrame([p for p in preds if p["config"] == cfg_name])\n            y = fd["label"].to_numpy(dtype=int)\n            s_run = fd["run_score"].to_numpy(dtype=float)\n            fold_rows.append(\n                {\n                    "config": cfg_name,\n                    "holdout_workload": "ALL",\n                    "roc_auc": safe_auc(y, s_run),\n                    "pr_auc": safe_ap(y, s_run),\n                    "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                    "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                    "n_features": int(fd["n_features"].iloc[0]),\n                }\n            )\n\n    pred_df = pd.DataFrame(preds).sort_values(["config", "workload", "stressor"])\n    fold_df = pd.DataFrame(fold_rows).sort_values(["config", "holdout_workload"])\n    diag_df = pd.DataFrame([{k: v for k, v in row.items() if not k.startswith("_")} for row in diagnostic_records]).sort_values(\n        ["config", "workload", "stressor"]\n    )\n\n    # Workload-conditioned score head: distance to workload nominal template.\n    pred_df["nominal_template_score"] = np.nan\n    pred_df["run_score_wc"] = pred_df["run_score"]\n    for cfg, d in pred_df.groupby("config"):\n        base = d[d["stressor"] == "NOMINAL"].set_index("workload")["run_score"].to_dict()\n        idx = d.index\n        pred_df.loc[idx, "nominal_template_score"] = d["workload"].map(base).to_numpy(dtype=float)\n        pred_df.loc[idx, "run_score_wc"] = np.abs(\n            pred_df.loc[idx, "run_score"].to_numpy(dtype=float)\n            - pred_df.loc[idx, "nominal_template_score"].to_numpy(dtype=float)\n        )\n\n    overall_rows = []\n    for cfg, d in pred_df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s_run = d["run_score"].to_numpy(dtype=float)\n        s_wc = d["run_score_wc"].to_numpy(dtype=float)\n        overall_rows.append(\n            {\n                "config": cfg,\n                "n_cases": int(len(d)),\n                "n_features": int(d["n_features"].iloc[0]),\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "fpr_run_alert": float(np.mean(d[d["label"] == 0]["run_alert"])),\n                "tpr_run_alert": float(np.mean(d[d["label"] == 1]["run_alert"])),\n                "median_nominal_score": float(np.median(d[d["label"] == 0]["run_score"])),\n                "median_anomaly_score": float(np.median(d[d["label"] == 1]["run_score"])),\n                "median_nominal_score_wc": float(np.median(d[d["label"] == 0]["run_score_wc"])),\n                "median_anomaly_score_wc": float(np.median(d[d["label"] == 1]["run_score_wc"])),\n            }\n        )\n    overall_df = pd.DataFrame(overall_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    fin = pred_df[pred_df["config"] == final_cfg]\n    stress_rows = []\n    neg = fin[fin["stressor"] == "NOMINAL"][["workload", "run_score", "run_score_wc"]].set_index("workload")\n    for a in ANOMALIES:\n        pos = fin[fin["stressor"] == a][["workload", "run_score", "run_score_wc"]].set_index("workload")\n        m = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n        y = np.array([0] * len(m) + [1] * len(m), dtype=int)\n        s_run = np.concatenate([m["run_score_neg"].to_numpy(dtype=float), m["run_score_pos"].to_numpy(dtype=float)])\n        s_wc = np.concatenate([m["run_score_wc_neg"].to_numpy(dtype=float), m["run_score_wc_pos"].to_numpy(dtype=float)])\n        stress_rows.append(\n            {\n                "stressor": a,\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "median_neg_score": float(np.median(m["run_score_neg"])),\n                "median_pos_score": float(np.median(m["run_score_pos"])),\n                "median_neg_score_wc": float(np.median(m["run_score_wc_neg"])),\n                "median_pos_score_wc": float(np.median(m["run_score_wc_pos"])),\n                "pos_neg_ratio": float((np.median(m["run_score_pos"]) + 1e-6) / (np.median(m["run_score_neg"]) + 1e-6)),\n                "pos_neg_diff": float(np.median(m["run_score_pos"]) - np.median(m["run_score_neg"])),\n                "pos_neg_ratio_wc": float((np.median(m["run_score_wc_pos"]) + 1e-6) / (np.median(m["run_score_wc_neg"]) + 1e-6)),\n                "pos_neg_diff_wc": float(np.median(m["run_score_wc_pos"]) - np.median(m["run_score_wc_neg"])),\n            }\n        )\n    stress_df = pd.DataFrame(stress_rows).sort_values("stressor")\n\n    mm_pr = float(np.mean(stress_df["pr_auc"]))\n    mm_roc = float(np.mean(stress_df["roc_auc"]))\n    mm_pr_wc = float(np.mean(stress_df["pr_auc_wc"]))\n    mm_roc_wc = float(np.mean(stress_df["roc_auc_wc"]))\n\n    filt = stress_df[~stress_df["stressor"].isin(["BRANCH", "TLB"])]\n    mm_pr_filt = float(np.mean(filt["pr_auc"]))\n    mm_roc_filt = float(np.mean(filt["roc_auc"]))\n    mm_pr_filt_wc = float(np.mean(filt["pr_auc_wc"]))\n    mm_roc_filt_wc = float(np.mean(filt["roc_auc_wc"]))\n\n    diag_pred_feature_df, diag_metrics_feature_df, diag_cm_feature = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_feature_contrib",\n    )\n    diag_pred_df, diag_metrics_df, diag_cm = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_mechanism_vector",\n    )\n    diag_tier_df = build_stressor_tier_contributions(diag_df, config=final_cfg)\n    mechanism_df = build_mechanism_summary(diag_df, config=final_cfg)\n    sequential_df = build_sequential_metrics(pred_df)\n    holdout_df = build_holdout_robustness_summary(fold_df)\n\n    pred_df.to_csv(out_dir / "case_predictions.csv", index=False)\n    fold_df.to_csv(out_dir / "fold_metrics.csv", index=False)\n    overall_df.to_csv(out_dir / "overall_metrics.csv", index=False)\n    stress_df.to_csv(out_dir / "stressor_metrics_final_config.csv", index=False)\n    diag_df.to_csv(out_dir / "case_diagnosis_summary.csv", index=False)\n    diag_pred_df.to_csv(out_dir / "stressor_diagnosis_predictions.csv", index=False)\n    diag_metrics_df.to_csv(out_dir / "stressor_diagnosis_metrics.csv", index=False)\n    diag_cm.to_csv(out_dir / "stressor_confusion_matrix.csv")\n    diag_tier_df.to_csv(out_dir / "stressor_tier_contributions.csv", index=False)\n    mechanism_df.to_csv(out_dir / "mechanism_group_summary.csv", index=False)\n    sequential_df.to_csv(out_dir / "sequential_metrics.csv", index=False)\n    holdout_df.to_csv(out_dir / "holdout_robustness_summary.csv", index=False)\n    diag_pred_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_predictions.csv", index=False)\n    diag_metrics_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_metrics.csv", index=False)\n    diag_cm_feature.to_csv(out_dir / "stressor_feature_confusion_matrix.csv")\n\n    overall_tex = overall_df[\n        ["config", "n_features", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "fpr_run_alert", "tpr_run_alert"]\n    ].rename(\n        columns={\n            "config": "Configuration",\n            "n_features": "Features",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "fpr_run_alert": "Run-FPR",\n            "tpr_run_alert": "Run-TPR",\n        }\n    )\n    stress_tex = stress_df[\n        ["stressor", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "pos_neg_ratio", "pos_neg_diff"]\n    ].rename(\n        columns={\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n            "pos_neg_diff": "Pos-Neg Score Delta",\n        }\n    )\n    (out_dir / "overall_metrics.tex").write_text(\n        to_latex_table(\n            overall_tex,\n            "DICE micro-twin + split-conformal run-level results under benign retraining.",\n            "tab:dice_full_overall",\n        )\n    )\n    (out_dir / "stressor_metrics_final_config.tex").write_text(\n        to_latex_table(\n            stress_tex,\n            "Final DICE configuration per-stressor separability.",\n            "tab:dice_full_stressor",\n        )\n    )\n    if not diag_metrics_df.empty:\n        diag_tex = diag_metrics_df.rename(\n            columns={\n                "config": "Configuration",\n                "n_cases": "Cases",\n                "top1_acc": "Top-1 Acc.",\n                "top2_acc": "Top-2 Acc.",\n                "macro_f1": "Macro-F1",\n                "mean_margin_to_second": "Mean Margin",\n            }\n        )\n        (out_dir / "stressor_diagnosis_metrics.tex").write_text(\n            to_latex_table(\n                diag_tex,\n                "Mechanism-group stressor attribution from DICE residual contributions across workloads.",\n                "tab:dice_stressor_diagnosis",\n            )\n        )\n    if not sequential_df.empty:\n        seq_tex = sequential_df.rename(\n            columns={\n                "config": "Configuration",\n                "benign_run_alert_rate": "Benign Run-Alert Rate",\n                "benign_persist_alerts_per_hour": "Benign Persist Alerts/hr",\n                "anomaly_detect_rate": "Anomaly Detect Rate",\n                "median_time_to_detect_s": "Median TTD (s)",\n                "detect_within_300s": "Detect <=300s",\n            }\n        )[\n            [\n                "Configuration",\n                "Benign Run-Alert Rate",\n                "Benign Persist Alerts/hr",\n                "Anomaly Detect Rate",\n                "Median TTD (s)",\n                "Detect <=300s",\n            ]\n        ]\n        (out_dir / "sequential_metrics.tex").write_text(\n            to_latex_table(\n                seq_tex,\n                "Sequential decision metrics for the DICE run-level detector.",\n                "tab:dice_sequential_metrics",\n            )\n        )\n\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config.png", score_col="run_score", title_tag="base")\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config_wc.png", score_col="run_score_wc", title_tag="workload-conditioned")\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot.png",\n        score_col="run_score",\n        y_label="Run score (base)",\n        title_tag="base",\n    )\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        score_col="run_score_wc",\n        y_label="Run score (workload-conditioned)",\n        title_tag="workload-conditioned",\n    )\n    plot_confusion_heatmap(\n        diag_cm,\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        title="Final-config prototype stressor attribution",\n    )\n    plot_stressor_tier_shares(\n        diag_tier_df,\n        fig_dir / "fig_stressor_tier_contributions.png",\n    )\n    plot_mechanism_shares(\n        mechanism_df,\n        fig_dir / "fig_mechanism_group_summary.png",\n    )\n    plot_detection_latency(\n        sequential_df,\n        fig_dir / "fig_detection_latency.png",\n    )\n\n    md = []\n    md.append("# DICE Full Retrain Results")\n    md.append("")\n    md.append("## Setup")\n    md.append(\n        f"- Protocol: {args.protocol}, benign-only fit/calibration, block_B={args.block_B}, "\n        f"alpha={args.alpha}, persist_k={args.persist_k}, gain={args.gain}"\n    )\n    md.append("")\n    md.append("## Overall")\n    append_text_table(md, overall_df)\n    md.append("")\n    md.append("## Final Config Stressors")\n    append_text_table(md, stress_df)\n    md.append("")\n    md.append("## Paper-style Aggregates (Final Config)")\n    md.append(f"- Base score mean stressor AUC-PR (all five): **{mm_pr:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (all five): **{mm_roc:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (all five): **{mm_pr_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (all five): **{mm_roc_wc:.4f}**")\n    md.append(f"- Base score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt_wc:.4f}**")\n    md.append("")\n    if not diag_metrics_df.empty:\n        md.append("## Diagnosis")\n        md.append("- Primary diagnosis uses mechanism-group centroids over workload-held residual summaries.")\n        append_text_table(md, diag_metrics_df)\n        md.append("")\n        if not diag_tier_df.empty:\n            md.append("## Final Config Tier Contribution Summary")\n            append_text_table(md, diag_tier_df)\n            md.append("")\n        if not mechanism_df.empty:\n            md.append("## Final Config Mechanism Summary")\n            append_text_table(md, mechanism_df)\n            md.append("")\n    if not sequential_df.empty:\n        md.append("## Sequential Decisioning")\n        append_text_table(md, sequential_df)\n        md.append("")\n    if not holdout_df.empty:\n        md.append("## Holdout Robustness (Workload Drift Proxy)")\n        append_text_table(md, holdout_df)\n        md.append("")\n    md.append("## Files")\n    for p in [\n        out_dir / "overall_metrics.csv",\n        out_dir / "stressor_metrics_final_config.csv",\n        out_dir / "sequential_metrics.csv",\n        out_dir / "case_diagnosis_summary.csv",\n        out_dir / "stressor_diagnosis_metrics.csv",\n        out_dir / "mechanism_group_summary.csv",\n        out_dir / "stressor_confusion_matrix.csv",\n        out_dir / "stressor_tier_contributions.csv",\n        out_dir / "overall_metrics.tex",\n        out_dir / "stressor_metrics_final_config.tex",\n        out_dir / "stressor_diagnosis_metrics.tex",\n        out_dir / "sequential_metrics.tex",\n        fig_dir / "fig_roc_pr_by_config.png",\n        fig_dir / "fig_roc_pr_by_config_wc.png",\n        fig_dir / "fig_run_score_boxplot.png",\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        fig_dir / "fig_stressor_tier_contributions.png",\n        fig_dir / "fig_mechanism_group_summary.png",\n        fig_dir / "fig_detection_latency.png",\n    ]:\n        md.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(md) + "\\n")\n\n    print(f"[OK] wrote results to: {out_dir}")\n    print("[OK] overall metrics:")\n    print(overall_df.to_string(index=False))\n    print("[OK] final config stressor metrics:")\n    print(stress_df.to_string(index=False))\n    if not diag_metrics_df.empty:\n        print("[OK] stressor diagnosis metrics:")\n        print(diag_metrics_df.to_string(index=False))\n    if not sequential_df.empty:\n        print("[OK] sequential metrics:")\n        print(sequential_df.to_string(index=False))\n    print(\n        "[OK] aggregates (base): "\n        f"all(AUC-PR={mm_pr:.4f}, ROC-AUC={mm_roc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt:.4f}, ROC-AUC={mm_roc_filt:.4f})"\n    )\n    print(\n        "[OK] aggregates (workload-conditioned): "\n        f"all(AUC-PR={mm_pr_wc:.4f}, ROC-AUC={mm_roc_wc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt_wc:.4f}, ROC-AUC={mm_roc_filt_wc:.4f})"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'


def _load_notebook_module(name: str, source: str) -> dict[str, object]:
    fake_file = REPO_ROOT / '__notebook__' / f'{name}.py'
    module = types.ModuleType(name)
    module.__file__ = str(fake_file)
    sys.modules[name] = module
    exec(source, module.__dict__)
    return module.__dict__


ANALYSIS_MODULE = _load_notebook_module('dice_generate_results_analysis_inline', NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE)
FULL_MODULE = _load_notebook_module('dice_train_eval_dice_pipeline_inline', NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE)

STAGE1_GAINS = [0.15, 0.25, 0.35, 0.50]
STAGE1_BLOCKS = [30, 60, 90, 120]
STAGE2_ALPHAS = [0.01, 0.02, 0.05, 0.10]
STAGE2_PERSISTS = [1, 2, 3, 5]


def deterministic_env() -> dict[str, str]:
    env = portable_env()
    os.environ.update(env)
    return env


def _run_module_main(module_ns: dict[str, object], argv: list[str]) -> None:
    argv_backup = sys.argv[:]
    try:
        sys.argv = argv
        module_ns['main']()
    finally:
        sys.argv = argv_backup


def ensure_dataset_root(root: Path) -> None:
    needed = [
        root / 'tier0',
        root / 'tier1_alt',
        root / 'tier2',
        root / 'no_nan_report.json',
    ]
    missing = [str(p) for p in needed if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Dataset root is missing required files/folders: {missing}')


def run_analysis_notebook(root: Path, source_hz: int = 5, out_dir: Path | None = None) -> Path:
    resolved_out = out_dir or (root / 'results_analysis')
    _run_module_main(
        ANALYSIS_MODULE,
        [
            'generate_results_analysis.py',
            '--root',
            str(root),
            '--source_hz',
            str(source_hz),
            '--out_dir',
            str(resolved_out),
        ],
    )
    return resolved_out


def run_full_notebook(
    root: Path,
    protocol: str,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    out_dir: Path | None = None,
) -> Path:
    resolved_out = out_dir or (root / ('results_dice_full_holdout' if protocol == 'workload_holdout' else 'results_dice_full'))
    _run_module_main(
        FULL_MODULE,
        [
            'train_eval_dice_pipeline.py',
            '--root',
            str(root),
            '--protocol',
            protocol,
            '--source_hz',
            str(source_hz),
            '--fit_ratio',
            str(fit_ratio),
            '--block_B',
            str(block_B),
            '--alpha',
            str(alpha),
            '--persist_k',
            str(persist_k),
            '--gain',
            str(gain),
            '--ridge_lambda',
            str(ridge_lambda),
            '--out_dir',
            str(resolved_out),
        ],
    )
    return resolved_out


def run_tuning_notebook(
    root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    ridge_lambda: float = 1e-3,
) -> Path:
    out_tune = root / 'results_dice_tuning'
    out_runs = out_tune / 'runs'
    out_tune.mkdir(parents=True, exist_ok=True)
    out_runs.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict[str, object]] = []
    alpha_fixed = 0.05
    persist_fixed = 3

    for gain in STAGE1_GAINS:
        for block_B in STAGE1_BLOCKS:
            tag = f'g{gain}_B{block_B}_a{alpha_fixed}_k{persist_fixed}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=block_B,
                    alpha=alpha_fixed,
                    persist_k=persist_fixed,
                    gain=gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'ok',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'error',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    stage1 = pd.DataFrame([r for r in summary_rows if r.get('stage') == 'gain_block' and r.get('status') == 'ok'])
    if stage1.empty:
        pd.DataFrame(summary_rows).to_csv(out_tune / 'sweep_summary.csv', index=False)
        raise RuntimeError('Notebook-local tuning stage 1 produced no successful runs.')

    best = stage1.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False).iloc[0]
    best_gain = float(best['gain'])
    best_block = int(best['block_B'])

    for alpha in STAGE2_ALPHAS:
        for persist_k in STAGE2_PERSISTS:
            tag = f'g{best_gain}_B{best_block}_a{alpha}_k{persist_k}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=best_block,
                    alpha=alpha,
                    persist_k=persist_k,
                    gain=best_gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'ok',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'error',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_tune / 'sweep_summary.csv', index=False)
    summary[(summary['stage'] == 'gain_block') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage1_gain_block.csv', index=False)
    summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage2_alpha_persist.csv', index=False)

    stage2 = summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].copy()
    stage2 = stage2.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False)
    recommended = stage2.iloc[0] if not stage2.empty else best
    recommendation = pd.DataFrame([
        {
            'gain': float(recommended['gain']),
            'block_B': int(recommended['block_B']),
            'alpha': float(recommended['alpha']),
            'persist_k': int(recommended['persist_k']),
            'pr_auc_wc': float(recommended['pr_auc_wc']),
            'roc_auc_wc': float(recommended['roc_auc_wc']),
        }
    ])
    recommendation.to_csv(out_tune / 'recommended_config.csv', index=False)
    return out_tune


def dataset_tree_sha256(root: Path) -> dict[str, object]:
    hasher = hashlib.sha256()
    count = 0
    for path in sorted(p for p in root.rglob('*') if p.is_file()):
        rel = path.relative_to(root).as_posix()
        if rel.split('/', 1)[0].startswith('results_'):
            continue
        hasher.update(rel.encode('utf-8'))
        with path.open('rb') as handle:
            while True:
                chunk = handle.read(1024 * 1024)
                if not chunk:
                    break
                hasher.update(chunk)
        count += 1
    return {'file_count': count, 'sha256': hasher.hexdigest()}


def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def package_versions() -> dict[str, str]:
    packages = [
        'matplotlib',
        'numpy',
        'pandas',
        'psutil',
        'scikit-learn',
        'scipy',
        'joblib',
        'threadpoolctl',
        'python-dateutil',
        'pytz',
        'tzdata',
    ]
    return {pkg.replace('-', '_'): importlib.metadata.version(pkg) for pkg in packages}


def write_run_manifest(
    root: Path,
    analysis_out: Path,
    full_out: Path,
    holdout_out: Path | None,
    tuning_out: Path | None,
) -> Path:
    manifest_dir = root / 'results_portable'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / 'run_manifest.json'

    env_yml = REPO_ROOT / 'environment.yml'
    req_txt = REPO_ROOT / 'requirements.txt'

    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'platform': platform.platform(),
        'python_version': sys.version.split()[0],
        'repo_root': str(REPO_ROOT),
        'dataset_root': str(root),
        'dataset_digest': dataset_tree_sha256(root),
        'environment_files': {
            'environment_yml': {'path': str(env_yml), 'sha256': file_sha256(env_yml)},
            'requirements_txt': {'path': str(req_txt), 'sha256': file_sha256(req_txt)},
        },
        'package_versions': package_versions(),
        'stages': {
            'analysis': str(analysis_out),
            'full': str(full_out),
            'holdout': str(holdout_out) if holdout_out else None,
            'tuning': str(tuning_out) if tuning_out else None,
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest_path


def run_notebook_pipeline(
    dataset_root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    run_holdout: bool = True,
    include_tuning: bool = False,
) -> dict[str, object]:
    deterministic_env()
    root = Path(dataset_root).expanduser().resolve()
    ensure_dataset_root(root)

    analysis_out = run_analysis_notebook(root, source_hz=source_hz)
    full_out = run_full_notebook(
        root,
        protocol='global',
        source_hz=source_hz,
        fit_ratio=fit_ratio,
        block_B=block_B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
        ridge_lambda=ridge_lambda,
    )
    holdout_out = None
    if run_holdout:
        holdout_out = run_full_notebook(
            root,
            protocol='workload_holdout',
            source_hz=source_hz,
            fit_ratio=fit_ratio,
            block_B=block_B,
            alpha=alpha,
            persist_k=persist_k,
            gain=gain,
            ridge_lambda=ridge_lambda,
        )
    tuning_out = run_tuning_notebook(root, source_hz=source_hz, fit_ratio=fit_ratio, ridge_lambda=ridge_lambda) if include_tuning else None
    manifest_path = write_run_manifest(root, analysis_out, full_out, holdout_out, tuning_out)

    return {
        'analysis_out': str(analysis_out),
        'full_out': str(full_out),
        'holdout_out': str(holdout_out) if holdout_out else '',
        'tuning_out': str(tuning_out) if tuning_out else '',
        'manifest_path': str(manifest_path),
        'run_holdout': run_holdout,
        'include_tuning': include_tuning,
    }


## Run End-to-End

This is the main **execution** section.

When `RUN_END_TO_END=True`, the next code cell actually runs:
1. the tier-analysis artifact generation
2. the full DICE digital-twin training/evaluation pass
3. the optional workload-holdout robustness pass
4. the optional tuning sweep

The digital-twin control knobs in that cell are:
- `fit_ratio`: benign fit/calibration split
- `block_B`: decision-block length
- `alpha`: split-conformal false-alarm target
- `persist_k`: persistent-alert requirement
- `gain`: fixed-gain synchronization strength
- `ridge_lambda`: ridge regularization for the benign dynamics model


In [ ]:
RUN_END_TO_END = True
INCLUDE_TUNING = False
RUN_HOLDOUT = True

runtime_start = perf_counter()
notebook_run_summary = {}

if RUN_END_TO_END:
    notebook_run_summary = run_notebook_pipeline(
        dataset_root=DATASET_ROOT,
        source_hz=5,
        fit_ratio=0.6,
        block_B=60,
        alpha=0.05,
        persist_k=3,
        gain=0.35,
        ridge_lambda=1e-3,
        run_holdout=RUN_HOLDOUT,
        include_tuning=INCLUDE_TUNING,
    )
    print('Notebook-local pipeline summary:')
    display(pd.DataFrame([notebook_run_summary]))
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')

runtime_seconds = round(perf_counter() - runtime_start, 2)
runtime_summary = {
    'runtime_seconds': runtime_seconds,
    'runtime_minutes': round(runtime_seconds / 60.0, 2),
    'run_end_to_end': RUN_END_TO_END,
    'run_holdout': RUN_HOLDOUT,
    'include_tuning': INCLUDE_TUNING,
    'repo_root': str(REPO_ROOT),
    'dataset_root': str(DATASET_ROOT),
    **notebook_run_summary,
}
NOTEBOOK_RUNTIME.write_text(json.dumps(runtime_summary, indent=2))
print('Notebook runtime summary:')
display(pd.DataFrame([runtime_summary]))


## What Runs vs What Reads

To make the notebook easier to follow:
- **Notebook-Local Pipeline Engine**: code definitions only; nothing is executed yet.
- **Run End-to-End**: this is the cell that runs the embedded DICE backend and writes results to disk.
- **Everything after this point**: these sections mostly **read generated outputs** and assemble reviewer-facing tables, dashboards, and appendix artifacts.

So if you only want to know where the digital twin is executed, focus on **Run End-to-End** and the embedded `run_notebook_pipeline(...)` definitions above it.


## ITC Artifact Roadmap

The rest of the notebook is grouped into the main-paper and appendix outputs that reviewers usually look for:
- main-paper separability, calibrated reliability, diagnosis, and digital-twin dashboards
- appendix holdout robustness, prediction review tables, bootstrap confidence intervals, and reproducibility files


## Core Analysis Outputs

These are the tier-level analysis tables and figures used to describe separability and dataset coverage.


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

print('Overall metrics')
display(overall)

print('Per-stressor metrics')
display(stressor)

print('Workload summary')
display(workload)

print('Feature inventory')
display(features[['tier_name', 'n_features_common', 'n_features_union']])

print('Case quality snapshot')
display(quality.head())


In [ ]:
for path in [
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
    FIG / 'fig_af_timeseries_tier2.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Methodology-Oriented Full Results

These outputs align with the preferred methodology: benign-only modeling, sequential decisioning, mechanism-level diagnosis, and robustness across observation heads.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
mechanism = pd.read_csv(OUT_FULL / 'mechanism_group_summary.csv')
tier_contrib = pd.read_csv(OUT_FULL / 'stressor_tier_contributions.csv')

print('Overall full-pipeline metrics')
display(overall_full)

print('Sequential decision metrics')
display(sequential)

print('Mechanism-group diagnosis metrics')
display(diagnosis)

print('Mechanism-group summary by stressor')
display(mechanism)

print('Tier contribution summary by stressor')
display(tier_contrib)

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
seq_final = sequential[sequential['config'] == 'tier0_tier1_tier2'].iloc[0]
diag_final = diagnosis[diagnosis['config'] == 'tier0_tier1_tier2'].iloc[0]
print('Final config (Tier-0 + Tier-1 + Tier-2)')
print('ROC-AUC              :', round(float(row_final['roc_auc_wc']), 4))
print('AUC-PR               :', round(float(row_final['pr_auc_wc']), 4))
print('Benign alert rate    :', round(float(seq_final['benign_run_alert_rate']), 4))
print('Median time-to-detect:', round(float(seq_final['median_time_to_detect_s']), 2))
print('Top-1 diagnosis acc  :', round(float(diag_final['top1_acc']), 4))
print('Top-2 diagnosis acc  :', round(float(diag_final['top2_acc']), 4))


In [ ]:
for path in [
    OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png',
    OUT_FULL / 'figures' / 'fig_detection_latency.png',
    OUT_FULL / 'figures' / 'fig_mechanism_group_summary.png',
    OUT_FULL / 'figures' / 'fig_stressor_confusion_matrix.png',
    OUT_FULL / 'figures' / 'fig_stressor_tier_contributions.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Benign/Anomaly Separation and Prediction Review

These tables establish the core paper story before any more elaborate diagnosis: benign and anomaly runs should separate cleanly, and reviewers should be able to inspect representative detected anomalies, false alarms, and misses.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')

sep_summary = (
    case_pred.groupby(['config', 'label'], sort=False)['run_score_wc']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
)
sep_summary['label_name'] = sep_summary['label'].map({0: 'benign', 1: 'anomaly'})
sep_summary.to_csv(PAPER_FULL / 'benign_anomaly_separation_summary.csv', index=False)

review_cols = ['config', 'case_id', 'workload', 'stressor', 'label', 'run_score_wc', 'run_alert', 'min_pvalue', 'time_to_detect_s']
tp = case_pred[(case_pred['label'] == 1) & (case_pred['run_alert'] == 1)][review_cols].sort_values('run_score_wc', ascending=False).head(10)
fp = case_pred[(case_pred['label'] == 0) & (case_pred['run_alert'] == 1)][review_cols].sort_values('run_score_wc', ascending=False).head(10)
fn = case_pred[(case_pred['label'] == 1) & (case_pred['run_alert'] == 0)][review_cols].sort_values('run_score_wc', ascending=True).head(10)

tp.to_csv(APPENDIX_FULL / 'prediction_review_true_positives.csv', index=False)
fp.to_csv(APPENDIX_FULL / 'prediction_review_false_positives.csv', index=False)
fn.to_csv(APPENDIX_FULL / 'prediction_review_false_negatives.csv', index=False)

print('Benign/anomaly separation summary')
display(sep_summary[['config', 'label_name', 'count', 'mean', 'median', 'min', 'max']])

print('Top detected anomalies')
display(tp)
print('Top false alarms')
display(fp)
print('Missed anomalies')
display(fn)

display(Image(filename=str(OUT_FULL / 'figures' / 'fig_run_score_boxplot_wc.png')))
display(Image(filename=str(OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png')))


## Research Question: Can Split-Conformal Residual Thresholds Stay Reliable Without Per-Workload Tuning?

This section answers the core reliability question directly from the generated case-level outputs.

We evaluate whether a single benign-calibrated threshold can hold a target false-alarm budget across workloads without tuning thresholds separately for each workload.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
TARGET_ALPHA = 0.05

reliability_rows = []
for cfg, d in case_pred.groupby('config', sort=False):
    benign = d[d['label'] == 0].copy()
    anomaly = d[d['label'] == 1].copy()
    reliability_rows.append({
        'config': cfg,
        'target_alpha': TARGET_ALPHA,
        'benign_block_false_alarm_rate': benign['n_block_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_persist_false_alarm_rate': benign['n_persist_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_run_false_alarm_rate': benign['run_alert'].mean(),
        'anomaly_run_detection_rate': anomaly['run_alert'].mean(),
        'median_anomaly_time_to_detect_s': anomaly.loc[anomaly['run_alert'] == 1, 'time_to_detect_s'].median(),
    })

reliability = pd.DataFrame(reliability_rows)
reliability_by_workload = (
    case_pred[case_pred['label'] == 0]
    .groupby(['config', 'workload'], sort=False)
    .apply(
        lambda x: pd.Series({
            'target_alpha': TARGET_ALPHA,
            'benign_block_false_alarm_rate': x['n_block_alerts'].sum() / x['n_blocks'].sum(),
            'benign_persist_false_alarm_rate': x['n_persist_alerts'].sum() / x['n_blocks'].sum(),
            'benign_run_false_alarm_rate': x['run_alert'].mean(),
        }),
        include_groups=False,
    )
    .reset_index()
)

(OUT_PAPER / 'full').mkdir(parents=True, exist_ok=True)
reliability.to_csv(OUT_PAPER / 'full' / 'conformal_reliability_summary.csv', index=False)
reliability_by_workload.to_csv(OUT_APPENDIX / 'full' / 'conformal_reliability_by_workload.csv', index=False)

print('Conformal reliability summary')
display(reliability)

print('Benign false-alarm rate by workload (no per-workload tuning)')
display(reliability_by_workload)


## Edge Digital-Twin Variants

This table reframes the generated outputs as edge digital-twin variants:
- `Single-head global twin`: one compact benign twin per observation head.
- `Workload-conditioned twin`: same edge model plus workload-conditioned residual scoring.
- `Mechanism-diagnosis twin`: same edge model with grouped mechanism attribution.

This is still edge-friendly because the model family remains linear and compact; the extra complexity is in scoring and diagnosis, not in a large cloud model.


In [ ]:
diag = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
variant_rows = []
for _, row in overall_full.iterrows():
    cfg = row['config']
    drow = diag[diag['config'] == cfg].iloc[0]
    variant_rows.append({
        'config': cfg,
        'single_head_roc_auc': row['roc_auc'],
        'single_head_pr_auc': row['pr_auc'],
        'workload_conditioned_roc_auc': row['roc_auc_wc'],
        'workload_conditioned_pr_auc': row['pr_auc_wc'],
        'mechanism_top1_acc': drow['top1_acc'],
        'mechanism_top2_acc': drow['top2_acc'],
        'mechanism_macro_f1': drow['macro_f1'],
    })
variant_summary = pd.DataFrame(variant_rows)
variant_summary.to_csv(OUT_PAPER / 'full' / 'digital_twin_variant_summary.csv', index=False)
print('Digital twin variant summary')
display(variant_summary)


## Capability Comparison With Prior Work

This is a capability-based comparison, not an accuracy-based one, because the prior papers use different platforms, observability sources, and datasets.

The table is derived from the attached E-SCOUT, OCTANE, TRACK, and DICE draft materials and is intended for positioning in the paper or appendix.


In [ ]:
prior_work = pd.DataFrame([
    {
        'method': 'E-SCOUT',
        'observability': 'Chip counters / sensors',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'not reported',
        'statistical_thresholding': 'outlier scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'host/edge (+ optional cloud)',
    },
    {
        'method': 'OCTANE',
        'observability': 'Chip counters / sensors (PMU/MSR + sensors)',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'partial',
        'statistical_thresholding': 'telemetry anomaly scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'on-device / on-chip',
    },
    {
        'method': 'TRACK',
        'observability': 'Cross-platform telemetry representations',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'partial',
        'statistical_thresholding': 'representation-based scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'device/host (fleet)',
    },
    {
        'method': 'DICE (this notebook)',
        'observability': 'OS telemetry + OS-mediated proxies + optional profiling',
        'continuous_test': 'yes',
        'diagnosis_cues': 'yes',
        'update_support': 'yes',
        'statistical_thresholding': 'split-conformal residual thresholding',
        'target_false_alarm_control': 'yes, benign-calibrated',
        'runs_at': 'host edge; optional fleet',
    },
])
(OUT_APPENDIX / 'comparison').mkdir(parents=True, exist_ok=True)
prior_work.to_csv(OUT_APPENDIX / 'comparison' / 'prior_work_capability_comparison.csv', index=False)
prior_work.to_csv(OUT_PAPER / 'comparison_prior_work_capability_comparison.csv', index=False)
print('Prior work capability comparison')
display(prior_work)


## Workload/Software Drift Proxy and Appendix Bundles

`results_dice_full_holdout/` is the portable workload-holdout evaluation. In this notebook, it is the practical proxy for workload/software drift robustness.


In [ ]:
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    print('Holdout robustness summary')
    display(holdout)
else:
    print('No holdout summary found yet.')

for folder in [OUT_PAPER, OUT_APPENDIX]:
    print(f'\n{folder}')
    if folder.exists():
        files = sorted(str(p.relative_to(folder)) for p in folder.rglob('*') if p.is_file())
        display(pd.DataFrame({'file': files[:100]}))
    else:
        print('Missing:', folder)


## DICE-Specific Design-Space Evaluation

OCTANE-style ROC/PR sweeps are useful, but DICE should show something more digital-twin specific: how observability, reliability, diagnosis, and detection latency trade off across edge micro-twin heads.

This section builds:
- an observability-performance frontier
- a twin-gain table showing the benefit of workload-conditioned residual scoring
- a stressor-level reliability map for anomaly detectability and time-to-detect


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

paper_full = OUT_PAPER / 'full'
paper_fig = paper_full / 'figures'
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

cfg_label = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
}

frontier = (
    overall_full[['config', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc']]
    .merge(
        sequential[['config', 'anomaly_detect_rate', 'median_time_to_detect_s', 'detect_within_120s', 'detect_within_300s']],
        on='config',
    )
    .merge(diagnosis[['config', 'top1_acc', 'top2_acc', 'macro_f1']], on='config')
    .merge(
        reliability[['config', 'target_alpha', 'benign_block_false_alarm_rate', 'benign_persist_false_alarm_rate', 'benign_run_false_alarm_rate']],
        on='config',
    )
)
frontier = frontier.merge(
    case_pred.groupby('config', sort=False)['n_features'].median().rename('n_features').reset_index(),
    on='config',
)
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout_frontier = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    holdout_frontier = holdout_frontier.rename(
        columns={
            'mean_pr_auc': 'holdout_mean_pr_auc',
            'worst_pr_auc': 'holdout_worst_pr_auc',
            'mean_roc_auc': 'holdout_mean_roc_auc',
        }
    )
    frontier = frontier.merge(
        holdout_frontier[['config', 'holdout_mean_pr_auc', 'holdout_worst_pr_auc', 'holdout_mean_roc_auc']],
        on='config',
        how='left',
    )
else:
    frontier['holdout_mean_pr_auc'] = np.nan
    frontier['holdout_worst_pr_auc'] = np.nan
    frontier['holdout_mean_roc_auc'] = np.nan
frontier['label'] = frontier['config'].map(cfg_label).fillna(frontier['config'])
frontier['observability_depth'] = frontier['label'].str.count('/') + 1
frontier['wc_gain_roc_auc'] = frontier['roc_auc_wc'] - frontier['roc_auc']
frontier['wc_gain_pr_auc'] = frontier['pr_auc_wc'] - frontier['pr_auc']
frontier['reliability_margin'] = frontier['target_alpha'] - frontier['benign_block_false_alarm_rate']
frontier['joint_detection_diagnosis'] = frontier['anomaly_detect_rate'] * frontier['top1_acc']
frontier['edge_efficiency_pr_per_feature'] = frontier['pr_auc_wc'] / frontier['n_features']
frontier['portable_pr_auc'] = frontier['holdout_mean_pr_auc'].fillna(frontier['pr_auc_wc'])
frontier['portable_roc_auc'] = frontier['holdout_mean_roc_auc'].fillna(frontier['roc_auc_wc'])
frontier.to_csv(paper_full / 'digital_twin_frontier_summary.csv', index=False)

gain = frontier[
    [
        'config',
        'label',
        'wc_gain_roc_auc',
        'wc_gain_pr_auc',
        'reliability_margin',
        'joint_detection_diagnosis',
        'edge_efficiency_pr_per_feature',
    ]
]
gain.to_csv(paper_full / 'digital_twin_gain_summary.csv', index=False)

stress_reliability = (
    case_pred[case_pred['label'] == 1]
    .groupby(['config', 'stressor'], sort=False)
    .apply(
        lambda x: pd.Series(
            {
                'anomaly_detect_rate': x['run_alert'].mean(),
                'median_time_to_detect_s': x.loc[x['run_alert'] == 1, 'time_to_detect_s'].median(),
                'median_peak_block_score': x['peak_block_score'].median(),
            }
        )
        , include_groups=False
    )
    .reset_index()
)
stress_reliability['label'] = stress_reliability['config'].map(cfg_label).fillna(stress_reliability['config'])
stress_reliability.to_csv(paper_full / 'stressor_reliability_map.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter = axes[0].scatter(
    frontier['n_features'],
    frontier['portable_pr_auc'],
    s=frontier['joint_detection_diagnosis'].fillna(0.0) * 1800 + 140,
    c=frontier['reliability_margin'],
    cmap='viridis',
    edgecolor='black',
    linewidth=0.8,
)
for _, row in frontier.iterrows():
    axes[0].annotate(row['label'], (row['n_features'], row['portable_pr_auc']), textcoords='offset points', xytext=(6, 6))
axes[0].set_xlabel('Median active features per edge head')
axes[0].set_ylabel('Portable AUC-PR (holdout if available)')
axes[0].set_title('Observability-portability frontier')
cbar = fig.colorbar(scatter, ax=axes[0])
cbar.set_label('Reliability margin (target alpha - empirical block FAR)')

axes[1].bar(frontier['label'], frontier['wc_gain_pr_auc'], color=['#0f766e', '#1d4ed8', '#b45309'])
axes[1].axhline(0.0, color='black', linewidth=1.0)
axes[1].set_ylabel('AUC-PR gain over single-head twin')
axes[1].set_title('Digital-twin gain from workload-conditioned scoring')

fig.tight_layout()
frontier_png = paper_fig / 'fig_digital_twin_frontier.png'
fig.savefig(frontier_png, dpi=200, bbox_inches='tight')
plt.close(fig)

detect_map = stress_reliability.pivot(index='stressor', columns='label', values='anomaly_detect_rate').fillna(0.0)
ttd_map = stress_reliability.pivot(index='stressor', columns='label', values='median_time_to_detect_s')

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
im0 = axes[0].imshow(detect_map.values, aspect='auto', cmap='YlGn', vmin=0.0, vmax=1.0)
axes[0].set_xticks(range(len(detect_map.columns)), detect_map.columns, rotation=30, ha='right')
axes[0].set_yticks(range(len(detect_map.index)), detect_map.index)
axes[0].set_title('Stressor-level anomaly detect rate')
for i in range(detect_map.shape[0]):
    for j in range(detect_map.shape[1]):
        axes[0].text(j, i, f"{detect_map.iloc[i, j]:.2f}", ha='center', va='center', color='black')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

ttd_plot = ttd_map.fillna(ttd_map.max().max())
im1 = axes[1].imshow(ttd_plot.values, aspect='auto', cmap='magma_r')
axes[1].set_xticks(range(len(ttd_plot.columns)), ttd_plot.columns, rotation=30, ha='right')
axes[1].set_yticks(range(len(ttd_plot.index)), ttd_plot.index)
axes[1].set_title('Median time-to-detect (s) for detected anomalies')
for i in range(ttd_plot.shape[0]):
    for j in range(ttd_plot.shape[1]):
        value = ttd_map.iloc[i, j]
        text = 'NA' if pd.isna(value) else f"{value:.0f}"
        axes[1].text(j, i, text, ha='center', va='center', color='white')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
stressor_png = paper_fig / 'fig_stressor_reliability_map.png'
fig.savefig(stressor_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Digital twin frontier summary')
display(frontier[['label', 'n_features', 'portable_pr_auc', 'holdout_worst_pr_auc', 'reliability_margin', 'anomaly_detect_rate', 'median_time_to_detect_s', 'top1_acc', 'joint_detection_diagnosis']])

print('Digital twin gain summary')
display(gain)

print('Stressor-level reliability map')
display(stress_reliability)

display(Image(filename=str(frontier_png)))
display(Image(filename=str(stressor_png)))


## Tier-Correlation Dashboard

This section shows how the observation tiers interact inside the final DICE head. The goal is not only to identify the dominant tier, but also to show whether tiers are redundant, complementary, or stressor-specific in their evidence contribution.


In [ ]:
tier_corr, stressor_tier, tier_corr_png = render_tier_correlation_dashboard(OUT_FULL, PAPER_FULL, PAPER_FIG)
print('Tier-share correlation matrix')
display(tier_corr)
print('Mean tier evidence by stressor')
display(stressor_tier)
display(Image(filename=str(tier_corr_png)))


## Residual Evidence Concentration

OCTANE reports how frequently top features are used. For DICE, a more digital-twin-specific question is how much anomaly evidence is explained by the top-k residual contributors.

If most residual evidence is captured by a few features or mechanism groups, the twin is not only accurate but also interpretable and edge-friendly.


In [ ]:
case_diag = pd.read_csv(OUT_FULL / 'case_diagnosis_summary.csv')
anom_diag = case_diag[case_diag['label'] == 1].copy()
mech_cols = [
    'compute_contrib',
    'memory_io_contrib',
    'thermal_power_contrib',
    'scheduler_runtime_contrib',
    'platform_pressure_contrib',
]
anom_diag['total_evidence'] = anom_diag[mech_cols].sum(axis=1).replace(0.0, np.nan)

for k in range(1, 6):
    cols = [f'top_feature_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'feature_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

for k in range(1, 4):
    cols = [f'top_mechanism_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'mechanism_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

coverage_cols = [
    'feature_top1_coverage',
    'feature_top2_coverage',
    'feature_top3_coverage',
    'feature_top4_coverage',
    'feature_top5_coverage',
    'mechanism_top1_coverage',
    'mechanism_top2_coverage',
    'mechanism_top3_coverage',
]
evidence_frontier = anom_diag.groupby('config', sort=False)[coverage_cols].mean().reset_index()
evidence_frontier['label'] = evidence_frontier['config'].map(cfg_label).fillna(evidence_frontier['config'])
evidence_frontier.to_csv(paper_full / 'residual_evidence_concentration.csv', index=False)

stressor_evidence = (
    anom_diag[anom_diag['config'] == 'tier0_tier1_tier2']
    .groupby('stressor', sort=False)[
        [
            'feature_top1_coverage',
            'feature_top3_coverage',
            'feature_top5_coverage',
            'mechanism_top1_coverage',
            'mechanism_top2_coverage',
            'mechanism_top3_coverage',
        ]
    ]
    .mean()
    .reset_index()
)
stressor_evidence.to_csv(paper_full / 'stressor_residual_evidence_concentration.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
feature_k = [1, 2, 3, 4, 5]
mechanism_k = [1, 2, 3]
for _, row in evidence_frontier.iterrows():
    axes[0].plot(feature_k, [row[f'feature_top{k}_coverage'] for k in feature_k], marker='o', linewidth=2, label=row['label'])
    axes[1].plot(mechanism_k, [row[f'mechanism_top{k}_coverage'] for k in mechanism_k], marker='o', linewidth=2, label=row['label'])
axes[0].set_xlabel('Top-k residual features')
axes[0].set_ylabel('Mean anomaly evidence coverage')
axes[0].set_xticks(feature_k)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title('Residual evidence concentration by feature rank')
axes[1].set_xlabel('Top-k mechanism groups')
axes[1].set_ylabel('Mean anomaly evidence coverage')
axes[1].set_xticks(mechanism_k)
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title('Residual evidence concentration by mechanism rank')
axes[1].legend(loc='lower right')
fig.tight_layout()
coverage_png = paper_fig / 'fig_residual_evidence_concentration.png'
fig.savefig(coverage_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Residual evidence concentration across digital-twin heads')
display(evidence_frontier)

print('Final-head stressor evidence concentration')
display(stressor_evidence)

display(Image(filename=str(coverage_png)))


## Bootstrap Confidence Intervals

The deployed DICE head remains compact. The extra runtime is spent offline here, through bootstrap confidence intervals that make the reported metrics more defensible for the paper and appendix.


In [ ]:
BOOTSTRAP_SAMPLES = 1000
bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=BOOTSTRAP_SAMPLES, seed=0)
print('Bootstrap confidence intervals')
display(bootstrap_ci)
display(Image(filename=str(bootstrap_png)))


## Paper Figure Layout

This section assembles the previous results into three reviewer-facing composite figures:
- performance stack: separation, ranking, calibration, and time-to-detect
- explainability dashboard: tier attribution, mechanism attribution, and diagnosis confusion
- portability dashboard: observability frontier, holdout robustness, and bootstrap uncertainty


In [ ]:
performance_stack, performance_stack_png = render_paper_performance_stack(
    case_pred,
    overall_full,
    sequential,
    reliability,
    PAPER_FULL,
    PAPER_FIG,
)
tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)
if 'holdout' not in globals() or holdout.empty:
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
portability_summary, portability_png = render_portability_dashboard(
    frontier,
    holdout,
    bootstrap_ci,
    PAPER_FULL,
    PAPER_FIG,
)

print('Performance stack summary')
display(performance_stack)
print('Portability summary')
display(portability_summary)
display(Image(filename=str(performance_stack_png)))
display(Image(filename=str(explainability_png)))
display(Image(filename=str(portability_png)))


## Optional LLM-Ready Case Cards

The detector itself is not an LLM. DICE remains a benign-trained digital twin with residual scoring and conformal decisioning.

This export is only a structured appendix artifact for optional later-stage diagnostics or reviewer summaries. It is model-agnostic. If you want a local summarizer later, suitable small instruct models include `Qwen2.5-7B-Instruct` or `Llama-3.1-8B-Instruct`, grounded only on the exported case cards.


In [ ]:
llm_cards = export_llm_case_cards(OUT_FULL, APPENDIX_FULL)
print('LLM-ready reviewer cards')
display(llm_cards.head(10))


In [ ]:
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
if NOTEBOOK_RUNTIME.exists():
    runtime_summary = json.loads(NOTEBOOK_RUNTIME.read_text())
    print('Notebook runtime summary:', runtime_summary)
print('Main paper artifacts:', PAPER_FULL)
print('Appendix artifacts :', APPENDIX_FULL)
